<center><font size=8>Airlines Q&A Bot</font></center>

# **Business Context**

Modern organizations operate in an environment where internal policies have grown increasingly complex, covering areas such as leaves, reimbursements, travel, hybrid work norms, performance management, compliance, and more. These policies are commonly documented in static, lengthy HR handbooks or PDFs, which employees often find difficult to understand, search, or apply in their daily work scenarios.

As a result:

- HR teams are burdened with repetitive queries about standard policies, diverting their focus from strategic initiatives.
- Employees struggle with confusion or non-compliance, as the dense and static format of policy documents obscures the necessary guidance.
- Both sides suffer from reduced productivity, HR spends excessive time addressing routine questions, while employees experience delays in obtaining the information they need to perform efficiently.

Addressing these challenges requires modern HR systems that centralize policy information, simplify access, and deliver clear, actionable insights. By leveraging technology to streamline policy communication and automate routine queries, organizations can enhance clarity, boost compliance, and ultimately improve overall operational efficiency.

# **Objective**

The goal is to develop a prototype that demonstrates how Natural Language Processing (NLP), powered by Retrieval-Augmented Generation (RAG), can help employees efficiently query company HR policies and receive accurate, context-aware, and easily understandable answers. Specifically, the system aims to:

- Answer employee questions by retrieving relevant content from official HR handbooks and policy documents.
- Handle ambiguous queries and follow-up questions by clarifying intent and distinguishing between similar policy categories (e.g., sick leave versus casual leave).
- Personalize responses based on role, location, or department, acknowledging that policies may differ (e.g., field staff versus HQ).
- Increase trust and compliance by citing sources (document name, section, and clause) for each response

# **Questions to Answer**

- What are the effects on the benefits I receive if my probation is extended?
- There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?
- What should I do if I notice suspected harassment with my female colleague?

# **Data Dictionary**

The employee handbook is an internal reference published by Flykite Airlines that outlines a wide range of workplace policies, guidelines, and procedures for staff. The handbook is provided in PDF format and serves as a comprehensive resource for employees across different roles within the airline.

# **Installing and Importing the Necessary Libraries**

In [1]:
!python -m pip install -q tiktoken==0.6.0 pypdf==4.0.1 langchain==0.1.1 langchain-community==0.0.13 chromadb==0.4.22 sentence-transformers==2.3.1


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!python -m pip install pandas


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import json
import tiktoken 

import pandas as pd

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings
)
from langchain_community.vectorstores import Chroma

# **Loading the Large Language Model**

In [4]:
LLM_MODEL = "llama2:13b"
OLLAMA_URL = "http://localhost:11434/api/generate"

In [5]:
import requests

#testing if the LLM was loaded successfully
test = requests.post(
    OLLAMA_URL,
    json={
        "model": LLM_MODEL,
        "prompt": "Hello",
        "stream": False
    }
)

print("LLM was loaded successfully")

LLM was loaded successfully


# **CRITERIA 1: QUESTION ANSWERING USING LLM**

## **Function to define the Model parameters & generate response**

In [6]:
#define function to process,define & respond to the prompt

def generate_llama_response(user_prompt):

    # System message
    system_message = """
    [INST]<<SYS>>Respond to the user question using the user prompt<</SYS>>[/INST]
    """
    
    # Combine user_prompt and system_message to create the prompt
    prompt = f"{user_prompt}\n{system_message}"

    OLLAMA_URL = "http://localhost:11434/api/generate"

    # Generate a response from the LLaMA model
    response = {

    "prompt" : prompt,         
    "model" : LLM_MODEL,
    "temperature" : 0.01,       #controls randomness of the output  
    "top_p" : 0.95,             #limits token selection to the ones with the mentioned probability
    "max_tokens" : 800,         #max number of tokens the model can generate in the response
    "top_k" : 50,               #limits the number of relevant tokens selection
    "repeat_penalty" : 1.2,     #penalizes repeated phrases in generated text
    "echo" : False,             #prevents the prompt from being repeated in the output
    "stream" : False            #the full response is returned at once instead of one by one
    }

    #Send a request to the Ollama API with the response in JSON format
    response = requests.post(OLLAMA_URL,json=response)

    #Convert the API response from JSON format into a Python dictionary
    result = response.json()

    #return only the generated text
    return result["response"]
    

In [7]:
questions = [ 
            "What are the effects on the benefits I receive if my probation is extended?",
            "There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?",
            "What should I do if I notice suspected harassment with my female colleague?"
            ]

In [8]:
#looping through each question & corresponding answer
for i,q in enumerate(questions,start=1):
    print(f"\nQuestion {i}: {q}\n")
    response = generate_llama_response(q)
    print("Answer:")
    print(response)
    print("\n")


Question 1: What are the effects on the benefits I receive if my probation is extended?

Answer:
   If your probation is extended, it may have several effects on the benefits you receive. Here are some possible consequences:

1. Extended Period of Supervision: An extension of your probation period means that you will be under supervision for a longer period. This can impact your daily life, as you will need to report regularly to your probation officer and adhere to specific rules and conditions.
2. Additional Requirements: Depending on the reasons for extending your probation, you may be required to fulfill additional requirements, such as completing more community service hours or attending additional counseling sessions. These extra requirements can add to your workload and stress levels.
3. Increased Fines or Fees: If your probation is extended, you may be subject to increased fines or fees. This can be a significant burden, especially if you are already struggling financially.
4.

## **Response Evaluation & Observations**

### QUESTION 1: What are the effects on the benefits I receive if my probation is extended?

#### Groundedness - 1/5

- The response isn't grounded with respect to the Flykite Airlines employee handbook.
-  Instead, it interpreted the question to be based out of a criminal probation rather than an employee probation within the airlines organization.
-  So the response involves matters relating to probation officers, drug testing, background checks for employment & housing, possession of firearms/alcohol etc.
-  This proves that without proper context with the company policy document, the model hallucinates and assumes a context of its own and generates an irrelevant response.

#### Relevance - 1/5

- The generated response is completely irrelevant to the Flykite Airlines employee handbook.
- While the question refers to the effect in employee benefits in the company due to extended probation, the answer deals with the effects of criminal probation.
- Instead of diving into topics such as job security, bonuses, hikes and other facilities and advantages an employee receives, it focuses on topics such as probation officers, drug testing, possession of firearms/alcohol, etc.
- This reaffirms the need of the Flykite Airlines employee handbook to improve relevance.

#### Observation

The response generated proves the importance of context in order to avoid hallucination. The response is irrelvant and not grounded. It has no insight into the employee handbook and so generates data which is already present in its training data. The model must be trained on the airlines company handbook in order to generate the required response.

### QUESTION 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

#### Groundedness - 2.5/5

- The response is definitely more grounded than the 1st question. But not enough since it's not based on the Flykite Airlines employee handbook.
- The model provides a more general response like "consult with your HR representative or your supervisor", which can apply to most of the corporate workplaces and not specifically Flyktie Airlines.
- The response is a bit vague since it informs the user that the specific policies and procedures may vary depending on the workplace and employer. 

#### Relevance - 4/5

- The response is definitely relevant to the question
- The model interprets the context correctly as a workplace scenario and provides the correct advice to reach out to the supervisor or HR representative.

#### Observation

- Though the response is more grounded and relevant, it is still pretty generic due to the lack of document knowledge.
- If the model had been trained on the specific airlines employee handbook, it would've provided information on more domain specific company policies like the number of leaves allowed and the official process for informing upper management.
- Also, the model could've avoided mentioning "Dear [User]" & "Best Regards, [Your Name]" and instead addressed the user as "Hey there" or anything sensitive. With this reponse, the model displays that it doesn't know the user's name and could've avoided this by giving a more neutral and generic greeting.
- While the model does display empathy and has the appropriate tone, due to the lack of document knowledge, it lacks procedural information and guidance.

### Question 3: What should I do if I notice suspected harassment with my female colleague?

#### Groundedness - 2/5

- The response is a bit grounded with the employee handbook, but not entirely.
- Although the model generated a response with mostly reasonable advice, it is still generic advice which isn't specific to the company policies.
- It doesn't reference any specific company policies and reporting procedures highlighted in the employee handbook.

#### Relevance - 4/5

- The response is highly relevant to the employee handbook.
- The model has grasped the correct context and generated a valid and practical response such as documenting the incidents and reporting to the supervisor/HR.

#### Observations

- The model has generated a slightly grounded and highly relevant response
- However, it misses to mention formal company procedures and policies due to lack of the employee handbook knowledge.
- Also, I feel it gave a slightly risky advice by asking the user to talk to the perpetrator. This isn't advisable since many corporate policies suggest to avoid directly confronting the perpetrator and report these kind of incident through official channels.

# CRITERIA 2: QUESTION ANSWERING USING LLM AND PROMPT ENGINEERING

## **Improved Prompt Engineering Function**

In [9]:
#define function to process,define & respond to the prompt

def generate_llama_response(user_prompt):

    # System message
    system_message = """
    [INST]<<SYS>>You are an HR assistant for Flykite Airlines.

    Your role is to help employees understand company HR policies
    and workplace procedures.

    Interpret the user prompt within the context of a corporate
    employee handbook and workplace environment.

    Provide responses that are:
    - Clear and professional
    - Relevant to employee HR policies
    - Helpful for workplace decision-making

    If a policy detail is unknown, provide general HR guidance
    rather than making assumptions.
    <</SYS>>[/INST]
    """
    
    # Combine user_prompt and system_message to create the prompt
    prompt = f"""

     ### SYSTEM INSTRUCTION ###
    {system_message}
    
    ### USER PROMPT ###
    {user_prompt}

    ### RESPONSE ###
    """

    OLLAMA_URL = "http://localhost:11434/api/generate"

    # Generate a response from the LLaMA model
    response = {

    "prompt" : prompt,         
    "model" : LLM_MODEL,
    "temperature" : 0.01,       #controls randomness of the output  
    "top_p" : 0.95,             #limits token selection to the ones with the mentioned probability
    "max_tokens" : 800,        #max number of tokens the model can generate in the response
    "top_k" : 50,               #limits the number of relevant tokens selection
    "repeat_penalty" : 1.2,     #penalizes repeated phrases in generated text
    "echo" : False,             #prevents the prompt from being repeated in the output
    "stream" : False            #the full response is returned at once instead of one by one
    }

    #Send a request to the Ollama API with the response in JSON format
    response = requests.post(OLLAMA_URL,json=response)

    #Convert the API response from JSON format into a Python dictionary
    result = response.json()

    #return only the generated text
    if "response" in result:
        return result["response"]
    else:
        print("API Error:",result)
        print("Error in generating response")
    
    

In [10]:
questions = [ 
            "What are the effects on the benefits I receive if my probation is extended?",
            "There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?",
            "What should I do if I notice suspected harassment with my female colleague?"
            ]

In [11]:
#looping through each question & corresponding answer
for i,q in enumerate(questions,start=1):
    print(f"\nQuestion {i}: {q}\n")
    response = generate_llama_response(q)
    print("Answer:")
    print(response)
    print("\n")


Question 1: What are the effects on the benefits I receive if my probation is extended?

Answer:

Hello! As an HR assistant for Flykite Airlines, I'd be happy to help you understand the effects of an extended probation period on your employee benefits.

According to our corporate employee handbook, during the probationary period (which can last up to 12 months), employees are eligible for a limited range of benefits. These benefits may include:

* Health insurance coverage through our group plan
* Paid time off (PTO) for vacation, sick leave, and personal days
* Access to our employee assistance program (EAP) for counseling and support services
* Participation in our 401(k) retirement savings plan with a company match

However, if your probation is extended beyond the initial 12-month period, you may lose access to some or all of these benefits. This is because the extension period is considered a "trial period" during which we assess your performance and suitability for long-term emp

## **Response Evaluation & Observations**

### QUESTION 1: What are the effects on the benefits I receive if my probation is extended?

#### Groundedness  -  2.5/5

- The response generated is much more improved & grounded than the previous one without prompting.
- The model doesn't hallucinate a completely wrong context and respond wrt a criminal probation. It interprets the employee handbook context correctly.
- But the response still lacks context wrt to the actual employee handbook. It provides generic HR guidelines rather than specify specific company policies and procedures to be followed.
- For eg, it informs the user that "your benefits may be impacted." & "you may not be eligible for certain benefits, such as health insurance or paid time off" & "you may be subject to additional performance expectations and evaluations" etc. These all seem like general HR policies which can be applied to any corporate.
- It doesn't list down any specific rule present in the Flykite Airlines handbook.

#### Relevance   -   4/5

- The response is highly relevant and aligned with the employee handbook much better than the previous attempt.
- However, the model's answer is still somewhat generic and lacks procedural details and rules.
- The response is professional and appropriate and is also encouraging to the employee who might need to undergo extended probation.

#### Observation

The response is more aligned with the corporate HR context compared to the previous one without prompt. But it still remains generic and lacks procedural detail. This limitation occurs because the model does not yet have access to the company handbook, which will be satisfied using RAG in the next criteria.

### QUESTION 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

#### Groundedness - 3/5

- The response is more slightly more grounded than the previous attempt.
- However, it is pretty obvious the model doesn't have access to the handbook since it advices the user check with the HR/supervisor as they will be able to provide him/her with more specific information about the leave policies and procedures, instead of providing the specific policies and procedures by itself.
- The response is again generic HR guidelines hallucinated as Flykite Airlines handbook policies.


#### Relevance   -   4/5

- The response is highly relevant.
- The response does appropriately addresses the employee's situation and also displays empathy, and suggests steps for notifying the organization and requesting leave.

#### Observation

- The response is slightly more grounded and is highly relevant. However, it remains generic still and doesn't mention the specific procedural details.
- On one note, it did improve its tone and addressed "Dear [Employee]" rather than "Dear [User]".
- The specific guidelines and policies can be mentioned once the model is trained on the Flykite Airlines handbook 

### Question 3: What should I do if I notice suspected harassment with my female colleague?

#### Groundedness - 3.5/5

- The response is slightly more grounded than the previous attempt.
- It aligns with common workplace harassment policies and emphasizes reporting procedures, documentation, confidentiality, etc.
- The previous response had been more empathetic and practical than professional. This response is more grounded wrt to professional employee handbook.
- However, it doesn't mention specific procedures or policies from the handbook and so still remains pretty generic rather than document oriented.

#### Relevance   -   4/5

- The response is highly relevant to the user's concern.
- It provides clear steps for reporting the harassment and also reassures the user about confidentiality and protection from retaliation. These words are appropriate and remains aligned with HR policies.
- It also retracted its previous risky advice regarding confronting the perpetrator directly in these scenarios. It even adviced the employee to avoid doing the same thing.

#### Observation

Compared to earlier responses, this response more professionally expresses formal workplace policy language. But it still lacks insight into company procedures or reporting channels, which would've been present if the model had access to the actual employee handbook.

# **CRITERIA 3: Data Preparation for RAG**

## Loading the Data File Provided

In [185]:
pdf_file = "Dataset - Flykite Airlines_HRP.pdf"

In [186]:
#loading the file using PyPDFLoader (since there's just one file)
pdf_loader = PyPDFLoader(pdf_file)

In [187]:
#Load the contents of the PDF into a list of documents
documents = pdf_loader.load()

In [188]:
#Print the no of pages loaded
print("Number of pages:",len(documents))

Number of pages: 14


## Split the data using a text splitter with the necessary attributes 

In [189]:
#Initializing the text splitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       #chunk size is a bit high since smaller chunks may split important policy explanations to multiple chunks
    chunk_overlap=100     #overlap is also high so that the context is retained between chunks containing important policies
)

In [190]:
#Load the PDF and split it into chunks simultaneously
document_chunks = pdf_loader.load_and_split(text_splitter)

In [191]:
#Printing the no of chunks created.
print("Number of chunks:",len(document_chunks))

Number of chunks: 34


In [19]:
#Looking at the first chunk

print(document_chunks[0])

page_content='Flykite\n \nAirlines:\n \nHuman\n \nResources\n \nPolicy\n \nHandbook\n \nIntroduction\n \nFlykite\n \nAirlines\n \nis\n \ndedicated\n \nto\n \ncultiv ating\n \nan\n \norganizational\n \ncultur e\n \nthat\n \nsyner gizes\n \noper ational\n \nexcellence\n \nwith\n \na\n \nsuppor tive,\n \nequitable,\n \nand\n \nlegally\n \ncompliant\n \nworkplace\n \nenvir onment\n \nacross\n \nall\n \ndepar tments\n \nand\n \nemplo yee\n \nlevels.\n \nThis\n \ndocument\n \nestablishes\n \nan\n \nexhaustiv e\n \nframework\n \ncomprising\n \nall\n \nhuman\n \nresour ce\n \npolicies\n \ncurrently\n \nin\n \neffect.\n \nAll\n \nprovisions\n \nare\n \nsubject\n \nto\n \namendment,\n \ninterpr etation,\n \nor\n \nrescindment\n \nat\n \nthe\n \nsole\n \ndiscr etion\n \nof\n \nthe\n \nHuman\n \nResour ces\n \nand\n \nLegal\n \ndepar tments.\n \nIn\n \nthe\n \nevent\n \nof\n \nambiguities\n \nor\n \nconﬂicting\n \ninterpr etations,\n \nthe\n \noﬃcial' metadata={'source': 'Dataset - Flykite Airline

Able to notice newline character between each word. This requires cleaning.

In [192]:
#Clean newline characters in each chunk
for doc in document_chunks:
    doc.page_content = doc.page_content.replace("\n", " ")

In [193]:
#Re-checking the first chunk

print(document_chunks[0])

page_content='Flykite   Airlines:   Human   Resources   Policy   Handbook   Introduction   Flykite   Airlines   is   dedicated   to   cultiv ating   an   organizational   cultur e   that   syner gizes   oper ational   excellence   with   a   suppor tive,   equitable,   and   legally   compliant   workplace   envir onment   across   all   depar tments   and   emplo yee   levels.   This   document   establishes   an   exhaustiv e   framework   comprising   all   human   resour ce   policies   currently   in   effect.   All   provisions   are   subject   to   amendment,   interpr etation,   or   rescindment   at   the   sole   discr etion   of   the   Human   Resour ces   and   Legal   depar tments.   In   the   event   of   ambiguities   or   conﬂicting   interpr etations,   the   oﬃcial' metadata={'source': 'Dataset - Flykite Airlines_HRP.pdf', 'page': 0}


The newline characters have been removed but able to notice double spaces between words. This requires cleaning.

In [194]:
import re

#Clean extracted text
for doc in document_chunks:

    #Remove multiple spaces
    text = re.sub(r"\s+", " ", doc.page_content)

    #Remove space before punctuation
    text = re.sub(r"\s([.,;:])", r"\1", text)

    #Save cleaned text
    doc.page_content = text.strip()


#Check chunk count
print("Number of chunks:", len(document_chunks))

#recheck first chunk
print(document_chunks[0])

Number of chunks: 34
page_content='Flykite Airlines: Human Resources Policy Handbook Introduction Flykite Airlines is dedicated to cultiv ating an organizational cultur e that syner gizes oper ational excellence with a suppor tive, equitable, and legally compliant workplace envir onment across all depar tments and emplo yee levels. This document establishes an exhaustiv e framework comprising all human resour ce policies currently in effect. All provisions are subject to amendment, interpr etation, or rescindment at the sole discr etion of the Human Resour ces and Legal depar tments. In the event of ambiguities or conﬂicting interpr etations, the oﬃcial' metadata={'source': 'Dataset - Flykite Airlines_HRP.pdf', 'page': 0}


In [195]:
#also checking consecutive chunk
print(document_chunks[1])

page_content='In the event of ambiguities or conﬂicting interpr etations, the oﬃcial determinations by these depar tments shall prevail and govern subsequent actions. 1. Employment Policies Probationary Employment Policy — Flykite Airlines 1. Duration of Initial Probation ● All new emplo yees are placed on a probationar y period of 90 calendar days from their oﬃcial start date. ● For technical, safety-critical, or senior management roles, probation is 120 calendar days. ● Any probation may be extended only once, for a maximum of 90 additional days, provided that a Performance Impr ovement Plan (PIP) is appr oved by' metadata={'source': 'Dataset - Flykite Airlines_HRP.pdf', 'page': 0}


As we can see there is some overlap between the chunks. This improves the coherence and relevance of retrieved results, as the model can better understand the relationship between adjacent parts of the document. It also helps in maintaining the flow of ideas and ensuring that critical context is available when generating answers, leading to more accurate and contextually consistent outputs.


## Load the embedding model

In [196]:
#choosing the GTE embedding model since it's designed for semantic retrieval task
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

print("The embedding model has loaded successfully")

The embedding model has loaded successfully


## Load the vector database

In [197]:
#defining name for the Chroma collection
report = 'Flykite_Airlines_HRP'

#Each doc chunk is converted into an embedding vector which is then stored in Chroma database
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    collection_name=report
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [198]:
#testing if the4 vector database got created
print(vectorstore)

The vector database has been successfully created

## Define the retriever with an appropriate search method and k value 

In [92]:
#Converts the Chroma vector store into a retriever for querying.
retriever = vectorstore.as_retriever(
    search_type='similarity',     # Specifies that retrieval is based on cosine similarity 
    search_kwargs={'k': 6}        # Retrieves the top 6 most similar documents for a given query.
)

## Retrieving the Relevant Documents

Let's ask a simple query and see what document chunks are returned based on the similarity search.

### Question 1: What are the effects on the benefits I receive if my probation is extended?

In [94]:
user_input = "What are the effects on the benefits I receive if my probation is extended?"

relevant_document_chunks = retriever.get_relevant_documents(user_input)

In [95]:
len(relevant_document_chunks)

6

In [96]:
for document in relevant_document_chunks:
    print(document.page_content.replace("\t", " "))

●
 
Extensions
 
are
 
granted
 
only
 
if
:
 
 
a.
 
The
 
emplo yee
 
has
 
achie ved
 
at
 
least
 
60%
 
of
 
their
 
probationar y
 
objectiv es.
 
 
b.
 
A
 
written
 
PIP
 
with
 
measur able
 
targets
 
is
 
issued
 
within
 
5
 
working
 
days
 
of
 
the
 
original
 
probation
 
end
 
date.
 
 
c.
 
The
 
Depar tment
 
Head
 
and
 
HR
 
Manager
 
both
 
sign
 
off
 
on
 
the
 
extension
 
request.
 
 
●
 
Emplo yees
 
will
 
be
 
notiﬁed
 
of
 
extensions
 
in
 
writing
 
at
 
least
 
7
 
calendar
 
days
 
befor e
 
the
 
probation
 
end
 
date.
 
3.
 
Impact
 
on
 
Beneﬁts,
 
Seniority ,
 
and
 
Contr act
 
●
 
While
 
on
 
probation
 
(including
 
any
 
extension),
 
emplo yees
 
are
 
not
 
eligible
 
for:
 
 
○
 
Annual
 
leave
 
encashment
 
 
○
 
Internal
 
role
● Extensions are granted only if: a. The emplo yee has achie ved at least 60% of their probationar y objectiv es. b. A written PIP with measur able targets is issued within 5 working days of the original probatio

The retrieved chunk contains the relevant information pertaining to the question and correctly addresses the effects of the benefits of an extended probation. It informs the employee that employees on probation are not eligible for:
- Annual leave encashment
- Internal role transfers
- Performance bonuses
- HR will assess contr act renewal eligibility based on performance history — no automatic carry-over is granted.

### Question 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave? 

In [31]:
user_input = "There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?"

relevant_document_chunks = retriever.get_relevant_documents(user_input)

In [32]:
len(relevant_document_chunks)

6

In [33]:
for document in relevant_document_chunks:
    print(document.page_content.replace("\t", " "))

family care, discr etionar y events) is suspended until investigation concludes. ○ Bereavement and jury duty exceptions still apply, but emplo yee must provide live contact details during leave for investigation-r elated communication. ○ Any leave exceeding 3 days must be appr oved directly by the Chief HR Oﬃcer. 7. Notiﬁcation Process ● Emplo yees must inform their direct super visor at least 48 hours befor e planned leave, except for emer gencies. ● For sudden emer gencies: ○ Verbal notiﬁcation within 6 hours of incident. vvaannii33@gmail.com 4C9X1MFLJ7 This file is meant for personal use by vvaannii33@gmail.com only.
● Jury duty or oﬃcial legal summons. ● Emer gency family care due to critical illness or accident. ● Natur al disasters affecting the emplo yee’s primar y residence. ● Other exceptional circumstances appr oved by HR and the Depar tment Head. 2. Entitlement & Duration Limits ● Bereavement: Up to 5 consecutiv e working days per incident. vvaannii33@gmail.com 4C9X1MFLJ7 Th

The retrieved chunk contains the relevant information pertaining to the question and correctly addresses the effects of the benefits of an extended probation. It contains information on Emergency family care & Special leave & Emergency absence.

### Question 3: What should I do if I notice suspected harassment with my female colleague? 

In [34]:
user_input = "What should I do if I notice suspected harassment with my female colleague? "

relevant_document_chunks = retriever.get_relevant_documents(user_input)

In [35]:
for document in relevant_document_chunks:
    print(document.page_content.replace("\t", " "))

be repor ted within 7 calendar days of occurr ence, even if the original harassment/discrimination case is still under review. 5. Investigation & Resolution Timelines ● HR must initiate a preliminar y review within 3 working days of receiving a repor t. ● Formal investigation begins within 7 working days and must conclude within 30 calendar days unless extended by written executiv e appr oval (max extension: 15 days). ● Both complainant and accused must receiv e: 1. Written acknowledgment of the complaint within 48 hours. 2. Outcome summar y and next steps within 5 working days of conclusion.
● Accepted repor ting channels: 1. Conﬁdential HR Helpline: +91-8000-456-789 (available Mon–Sat, 9 AM–8 PM IST). 2. Secur e Email Portal: hr.ethics@ﬂykiteair.com. 3. Anonymous drop-bo xes in crew lounges and staff cafeterias. ● Repor ts must include date, location, persons involved, and a brief incident description. 4. Protection Against Retaliation ● Retaliation (e.g., demotion, shift change, exc

The retrieved chunk contains the relevant information pertaining to the question and correctly addresses the effects of the benefits of an extended probation. It contains information on Investigation & Resolution Timelines, Harassment Policy and provides the confidential HR Helpline as well as a secure email portal.

# **CRITIERIA 4: Question Answering using RAG**

## Prompt Design

Prompt designing is a crucial part of designing a RAG based system, it consists mainly of two parts:

- system message: This is the instruction that has to be given to the LLM.
- user message template: This is a message template that contains the context retrieved from the document chunks and the User Query.

In [36]:
qna_system_message = """
You are an assistant whose work is to give answers to questions with repect to a context.
User input will have the context required by you to answer user questions.

This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Answer primarily using the information provided in the ###Context.
If the answer is not explicitly stated, you may infer it logically from the context.
Do not mention anything about the information in ###Context or the question in ###Question in your final answer.

If the answer is not explicitly stated, try to infer it logically from the context before saying "I don't know".

Remember that the answer to ###Question might not always be directly present in the information provided in the ###Context.
the answer can be indirectly derived from the information in ###Context.

"""

In [37]:
qna_user_message_template = """
Conider the following ###Context and ###Question
###Context
{context}

###Question
{question}
"""

## Importing the Necessary Libraries & Initializations for Langchain

In [97]:
!pip install langchain


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [98]:
#importing the necessary libraries for Langchain
from langchain.memory import ConversationBufferMemory
from langchain.schema import HumanMessage,AIMessage

In [99]:
#Stores all the user inputs and messages
memory = ConversationBufferMemory(return_messages=True)

## RAG Response Generation Function using Langchain

In [41]:
def RAG_with_memory(user_input,memory):
    """
    Args:
        user_input: User query
        llm: Language model
        memory: LangChain memory object to store conversation history
    
    Returns:
        Generated response using RAG + conversation memory
    """
    
    #Retrieve relevant document chunks from vector DB
    relevant_document_chunks = retriever.get_relevant_documents(user_input)
    
    #Extract and clean text from documents
    context_list = [d.page_content.replace("\t", " ") for d in relevant_document_chunks]
    
    #Combine all chunks into a single context string
    context_for_query = "\n\n".join(context_list)
    
    #Load previous conversation history from memory
    chat_history = memory.load_memory_variables({})["history"]
    
    #Convert structured messages into plain text
    history_text = ""
    for msg in chat_history:
        if isinstance(msg, HumanMessage):
            history_text += f"User: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            history_text += f"Assistant: {msg.content}\n"
    
    #Append conversation history to context
    full_context = f"""
###Conversation History:
{history_text}

###Context:
{context_for_query}
"""
    
    #Build prompt using the System message & full context(conversation history + context)
    prompt = f"""[INST]{qna_system_message}
    {{'user': {qna_user_message_template.format(context=full_context, question=user_input)} }}
    [/INST]"""

    OLLAMA_URL = "http://localhost:11434/api/generate"
    
    # Generate a response from the LLaMA model
    response = {

    "prompt" : prompt,         
    "model" : LLM_MODEL,
    "temperature" : 0.01,       #controls randomness of the output  
    "top_p" : 0.95,             #limits token selection to the ones with the mentioned probability
    "max_tokens" : 800,         #max number of tokens the model can generate in the response
    "top_k" : 50,               #limits the number of relevant tokens selection
    "repeat_penalty" : 1.2,     #penalizes repeated phrases in generated text
    "echo" : False,             #prevents the prompt from being repeated in the output
    "stream" : False            #the full response is returned at once instead of one by one

    }

    try:
        #Send a request to the Ollama API with the response in JSON format
         response = requests.post(OLLAMA_URL,json=response)

        #Convert the API response from JSON format into a Python dictionary
         result = response.json()
    #Handle errors
    except Exception as e:
        result = f"Error: {e}"
        
          
  #Save current interaction into memory
    memory.save_context(
        {"input": user_input},
        {"output": result["response"]}
    )
    
    #return only the generated text
    return result["response"]

In [42]:
questions = [ 
            "What are the effects on the benefits I receive if my probation is extended?",
            "There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?",
            "What should I do if I notice suspected harassment with my female colleague?"
            ]

In [43]:
#looping through each question & corresponding answer
for i,q in enumerate(questions,start=1):    
    print(f"\nQuestion {i}: {q}\n")
    response = RAG_with_memory(q,memory)
    print("Answer:")
    print(response)
    print("\n")


Question 1: What are the effects on the benefits I receive if my probation is extended?

Answer:
 Based on the provided context, there are several restrictions and limitations on benefits for employees under probation. These include:

1. Annual leave encashment: Employees on probation are not eligible for annual leave encashment.
2. Internal role transfers: Employees on probation are not eligible for internal role transfers.
3. Performance bonuses: Employees on probation are not eligible for performance bonuses.
4. Seniority accrual: Seniority accrual is only available after successful probation completion.
5. Special leave requests: Special leave requests during probation are limited to maximum 2 consecutive working days for bereavement or emergency family care, and full jury duty leave is allowed if legally mandated, but the probation period is automatically extended by the same duration.

However, it is not explicitly stated in the context that the employee's benefits will be affec

## **Response Evaluation & Observations**

### QUESTION 1: What are the effects on the benefits I receive if my probation is extended?

#### Groundedness - 4/5

- The generated resonse, post RAG, is grounded and aligned closely with the policy document.
- The model avoids hallucination and states that it can't provide a definitive answer without further information.
- However, it does miss a few points like the effect on performance bonuses if the probation is extended. It has also failed to warn the employee that "If an employee has two or more extensions in different roles (due to internal transfers),HR will assess contract renewal eligibility based on performance history—no automatic carry-over is granted"

#### Relevance - 5/5

- The response is 100% relevant to the employee handbook.
- The response correctly mentions the effects on the benefits in case of probation and also avoids hallucination.

#### Observation

The model successfully interpreted the effects of probation extension. However, certain policy details were not included due to retrieval limitations, indicating that answer completeness depends heavily on the relevance and coverage of retrieved chunks.

### QUESTION 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

#### Groundedness - 4.5/5

- The answer is highly grounded to the Flykite Airlines HR guideline.
- The model took the context strictly out of the handbook and avoided deviation or hallucination.
- The structured & point format improves clarity and readability of the response.
- However, minor residual behavior such as the inclusion of generic disclaimers indicates that the model is not fully constrained to the retrieved context.

#### Relevance - 5/5

- The generated response is highly relevant to the policy document.
- The model highlights the policies using crisp points in a procedural manner.
- The model also avoids hallucination by stating that the policy document does not explicitly state if attendance at the last rites is covered under Bereavement leave and directs the employee towards the HR, rather than providing a generic answer.
- However, the model introduces generic disclaimers that are not required in a structured RAG setting, indicating residual behavior from previously trained responses.

#### Observations

The response is both highly grounded and relevant in its alignment to the Flykite Airlines HR guideline. The model maintains groundedness and avoids hallucination while clearly communicating procedural requirements. However, disclaimers can be avoided in this particular RAG setting. This indicates the necessity to further refine the prompt design to better constrain the model to context specific outputs.

### Question 3: What should I do if I notice suspected harassment with my female colleague?

#### Groundedness - 4/5

- The response is well grounded in the FlyKite Airlines HR policy and highlights the procedures outlined in the retrieved context.
- The model correctly identifies key steps such as reporting through appropriate channels, documentation timelines, and escalation procedures.
- It avoids hallucination and ensures that all major responses are aligned with the policy document.
- However, certain critical details, such as specific contact information (e.g., helpline number or official email ID) weren't included. This might be  due to incomplete retrieval of relevant document chunks.

#### Relevance - 5/5

- The response is highly relevant to the user’s query and directly addresses the actions to be taken in cases of suspected harassment.
- It presents the information in a clear & structured format, improving readability for the employee.
- The model focuses on policy related procedures and avoids introducing unrelated or misleading information.

#### Observations

The response clearly provides the correct policy guidelines & shows strong alignment with the FlyKite Airlines HR framework. The structured format enhances readability and ensures that the employee can easily follow the required procedures. However, the omission of specific contact details indicates a limitation in the retrieval stage, where not all relevant information was retrieved. This reinforces the dependency on retrieval quality in RAG systems.

## Check the Memory Retention

Checking the memory rentention of the model post Langchain integrating response generation function by asking the model follow up questions

In [44]:
question = "Okay, can I address the perpetrator who harassed my female colleague?"
print(f"\nQuestion: {question}\n")
response = RAG_with_memory(question,memory)
print("Answer:")
print(response)


Question: Okay, can I address the perpetrator who harassed my female colleague?

Answer:
 Based on the provided context, there are several restrictions and limitations on benefits for employees under probation. However, it is not explicitly stated in the context that the employee's benefits will be affected during an extended probation period. Therefore, I cannot provide a definitive answer to your question without further clarification or information.

Regarding your request to address the perpetrator who harassed your female colleague, you should consult with your supervisor and HR manager to report the incident and seek guidance on the next steps to take. It is important to document any evidence of the harassment and to avoid any retaliation against the accused employee. Additionally, you may want to consider speaking with a lawyer or a legal aid organization to understand your options and protect your rights.

Please note that while under active investigation for harassment/discri

In [45]:
question = "Okay, can I atleast talk to my female colleague?"
print(f"\nQuestion: {question}\n")
response = RAG_with_memory(question,memory)
print("Answer:")
print(response)


Question: Okay, can I atleast talk to my female colleague?

Answer:
 Based on the provided context, there are several restrictions and limitations on benefits for employees under probation. However, it is not explicitly stated in the context that the employee's benefits will be affected during an extended probation period. Therefore, I cannot provide a definitive answer to your question without further clarification or information.

Regarding your request to talk to your female colleague, you should consult with your supervisor and HR manager to report any incidents of suspected harassment and seek guidance on the next steps to take. It is important to document any evidence of the harassment and to avoid any retaliation against the accused employee. Additionally, you may want to consider speaking with a lawyer or a legal aid organization to understand your options and protect your rights.

Please note that while under active investigation for harassment/discrimination, non-mandatory s

### Observation

Though the first question was a simple follow up question, the 2nd one was clearly without context to the harrassment scenario. The model successfully proved its memory retention by interpreting the context of the question correctly. The response is aligned to the context in its groundedness and relevance. 
However, we can notice that  while the responses were largely grounded, it also showed signs of overgeneralization. For eg, the model inferred restrictions(avoiding interaction with the female colleague) that were not explicitly stated in the policy, indicating a blend of retrieved knowledge and pre-trained safety behavior.

# **CRITERIA 5: Question Answering using RAG Fine Tuning**

## Tuning #1

Since we had used a very low value for temperature(0.01), we felt that the answers were too rigid and over restricted. By increasing the value to 0.05, we can reduce the rigidity slightly

In [46]:
def RAG_with_memory(user_input,memory):
    """
    Args:
        user_input: User query
        llm: Language model
        memory: LangChain memory object to store conversation history
    
    Returns:
        Generated response using RAG + conversation memory
    """
    
    #Retrieve relevant document chunks from vector DB
    relevant_document_chunks = retriever.get_relevant_documents(user_input)
    
    #Extract and clean text from documents
    context_list = [d.page_content.replace("\t", " ") for d in relevant_document_chunks]
    
    #Combine all chunks into a single context string
    context_for_query = "\n\n".join(context_list)
    
    #Load previous conversation history from memory
    chat_history = memory.load_memory_variables({})["history"]
    
    #Convert structured messages into plain text
    history_text = ""
    for msg in chat_history:
        if isinstance(msg, HumanMessage):
            history_text += f"User: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            history_text += f"Assistant: {msg.content}\n"
    
    #Append conversation history to context
    full_context = f"""
###Conversation History:
{history_text}

###Context:
{context_for_query}
"""
    
    #Build prompt using the System message & full context(conversation history + context)
    prompt = f"""[INST]{qna_system_message}
    {{'user': {qna_user_message_template.format(context=full_context, question=user_input)} }}
    [/INST]"""

    OLLAMA_URL = "http://localhost:11434/api/generate"
    
    # Generate a response from the LLaMA model
    response = {

    "prompt" : prompt,         
    "model" : LLM_MODEL,
    "temperature" : 0.05,       #controls randomness of the output  
    "top_p" : 0.95,             #limits token selection to the ones with the mentioned probability
    "max_tokens" : 800,         #max number of tokens the model can generate in the response
    "top_k" : 50,               #limits the number of relevant tokens selection
    "repeat_penalty" : 1.2,     #penalizes repeated phrases in generated text
    "echo" : False,             #prevents the prompt from being repeated in the output
    "stream" : False            #the full response is returned at once instead of one by one

    }

    try:
        #Send a request to the Ollama API with the response in JSON format
         response = requests.post(OLLAMA_URL,json=response)

        #Convert the API response from JSON format into a Python dictionary
         result = response.json()
    #Handle errors
    except Exception as e:
        result = f"Error: {e}"
        
          
  #Save current interaction into memory
    memory.save_context(
        {"input": user_input},
        {"output": result["response"]}
    )
    
    #return only the generated text
    return result["response"]

In [47]:
questions = [ 
            "What are the effects on the benefits I receive if my probation is extended?",
            "There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?",
            "What should I do if I notice suspected harassment with my female colleague?"
            ]

In [48]:
#looping through each question & corresponding answer
for i,q in enumerate(questions,start=1):    
    print(f"\nQuestion {i}: {q}\n")
    response = RAG_with_memory(q,memory)
    print("Answer:")
    print(response)
    print("\n")


Question 1: What are the effects on the benefits I receive if my probation is extended?

Answer:
Based on the provided context, there are several restrictions and limitations on benefits for employees under probation. These include:

1. Annual leave encashment: Employees on probation are not eligible for annual leave encashment.
2. Internal role transfers: Employees on probation are not eligible for internal role transfers.
3. Performance bonuses: Employees on probation are not eligible for performance bonuses.
4. Seniority accrual: Seniority accrual is only available after successful probation completion.
5. Special leave requests: Special leave requests during probation are limited to maximum 2 consecutive working days for bereavement or emergency family care, and full jury duty leave is allowed if legally mandated, but the probation period is automatically extended by the same duration.

However, it is not explicitly stated in the context that the employee's benefits will be affect

### **Response Evaluation & Observations**

#### QUESTION 1: What are the effects on the benefits I receive if my probation is extended?

##### Groundedness - 4.5/5

- Post tuning the parameter temperature to the value 0.05 in order to decrease rigidity, it retains its groundedness and aligned closely with the policy document.
- The model avoids hallucination and states that it can't provide a definitive answer without further information.
- The response is slightly more grounded as it mentions that the Performance Bonuses will be affected. It had missed to recall the same in pre-fine tuning. However, the model has still not warned the employee that "If an employee has two or more extensions in different roles (due to internal transfers),HR will assess contract renewal eligibility based on performance history—no automatic carry-over is granted"
- However, the use of the generic disclaimer continues. This requires modification in the prompt

##### Relevance - 5/5

- Though we have increased the randomness of the generated response, it remains 100% relevant to the employee handbook.
- The response correctly mentions the effects on the benefits in case of probation and avoids hallucination.

##### Observation

Despite fine tuning the reponse generation function to increase the randomness, the model has displayed even slightly higher groundedness & retained high relevance as compared to pre fine tuning. The response also includes one of the two missed information regarding the effect on Performance Bonus, highlighting the increase in groundedness. 
However, the continuation of generic disclaimer must be given attention to.

#### QUESTION 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

##### Groundedness - 5/5

- The response remains highly grounded to the Flykite Airlines HR guideline.
- Post allowing flexibility in the randomness of the generated response, the model continued to retrieve the context out of the handbook and avoided deviation or hallucination.
- The response has also improved as it has become less rigid and more user friendly. Unlike the previous time, it doesn't suggest the employee to provide live contact details for investigative purposes for this scenario.
- The model also excludes the generic disclaimer for this scenario, reaffirming more human-like flexibility as compared to the previous attempt.

##### Relevance - 4.5/5

- The generated response remains highly relevant to the policy document.
- The model continues avoiding hallucination by stating that the policy document does not explicitly state if attendance at the last rites is covered under Bereavement leave and directs the employee towards the HR, rather than providing a generic answer.
- While readability improved, the model introduced generalized statements regarding other companies which wasn't present in the policy document.

##### Observation

The response remains both highly grounded and relevant in its alignment to the Flykite Airlines HR guideline. The model maintains groundedness and avoids hallucination while clearly communicating procedural requirements. The model displays more flexibility and less rigidity and has also ommited the generic disclaimer which was not apt for the RAG setting. However, the model has introduced generalized statements not present in the policy document, indicating a slight trade-off between flexibility and strict grounding.

#### Question 3: What should I do if I notice suspected harassment with my female colleague?

##### Groundedness - 4.5/5

- The response continues to be grounded in the FlyKite Airlines HR policy and highlights the procedures outlined in the retrieved context.
- The model again identifies key steps such as reporting through appropriate channels, documentation timelines, and escalation procedures.
- It avoids hallucination and ensures that all major responses are aligned with the policy document.
- Critical details such as Email Portal & the HR Helpline number has been provided post fine tuning. Previously, the model had missed to generate the same.

##### Relevance - 5/5

- Post fine tuning the temperature to 0.05, the response remains highly relevant to the user’s query and directly addresses the actions to be taken in cases of suspected harassment.
- It presents the information in a clear & structured format, improving readability for the employee.
- The model focuses on policy related procedures and avoids introducing unrelated or misleading information.

##### Observations

The response demonstrates improved completeness and grounding, successfully incorporating critical details such as helpline and email contact information. However, minor residual generic phrasing remain.

In [49]:
question = "Okay, can I atleast talk to my female colleague?"
print(f"\nQuestion: {question}\n")
response = RAG_with_memory(question,memory)
print("Answer:")
print(response)


Question: Okay, can I atleast talk to my female colleague?

Answer:
 Based on the provided context, there are several restrictions and limitations on benefits for employees under probation. However, it is not explicitly stated in the context that the employee's benefits will be affected during an extended probation period. Therefore, I cannot provide a definitive answer to your question without further clarification or information.

Regarding your request to talk to your female colleague, you should consult with your supervisor and HR manager to report any incidents of suspected harassment and seek guidance on the next steps to take. It is important to document any evidence of the harassment and to avoid any retaliation against the accused employee. Additionally, you may want to consider speaking with a lawyer or a legal aid organization to understand your options and protect your rights.

Please note that while under active investigation for harassment/discrimination, non-mandatory s

## Tuning #2

The fine tuning of temperature to 0.05 improved response flexibility while maintaining groundedness. To further enhance the model’s ability to explore relevant context, we have increased the temperature to 0.1. However, to prevent excessive randomness and maintain response coherence, the top_p value is reduced from 0.95 to 0.90. This ensures a controlled balance between diversity and precision in the generated responses.

In [50]:
def RAG_with_memory(user_input,memory):
    """
    Args:
        user_input: User query
        llm: Language model
        memory: LangChain memory object to store conversation history
    
    Returns:
        Generated response using RAG + conversation memory
    """
    
    #Retrieve relevant document chunks from vector DB
    relevant_document_chunks = retriever.get_relevant_documents(user_input)
    
    #Extract and clean text from documents
    context_list = [d.page_content.replace("\t", " ") for d in relevant_document_chunks]
    
    #Combine all chunks into a single context string
    context_for_query = "\n\n".join(context_list)
    
    #Load previous conversation history from memory
    chat_history = memory.load_memory_variables({})["history"]
    
    #Convert structured messages into plain text
    history_text = ""
    for msg in chat_history:
        if isinstance(msg, HumanMessage):
            history_text += f"User: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            history_text += f"Assistant: {msg.content}\n"
    
    #Append conversation history to context
    full_context = f"""
###Conversation History:
{history_text}

###Context:
{context_for_query}
"""
    
    #Build prompt using the System message & full context(conversation history + context)
    prompt = f"""[INST]{qna_system_message}
    {{'user': {qna_user_message_template.format(context=full_context, question=user_input)} }}
    [/INST]"""

    OLLAMA_URL = "http://localhost:11434/api/generate"
    
    # Generate a response from the LLaMA model
    response = {

    "prompt" : prompt,         
    "model" : LLM_MODEL,
    "temperature" : 0.1,        #controls randomness of the output  
    "top_p" : 0.90,             #limits token selection to the ones with the mentioned probability
    "max_tokens" : 800,         #max number of tokens the model can generate in the response
    "top_k" : 50,               #limits the number of relevant tokens selection
    "repeat_penalty" : 1.2,     #penalizes repeated phrases in generated text
    "echo" : False,             #prevents the prompt from being repeated in the output
    "stream" : False            #the full response is returned at once instead of one by one

    }

    try:
        #Send a request to the Ollama API with the response in JSON format
         response = requests.post(OLLAMA_URL,json=response)

        #Convert the API response from JSON format into a Python dictionary
         result = response.json()
    #Handle errors
    except Exception as e:
        result = f"Error: {e}"
        
          
  #Save current interaction into memory
    memory.save_context(
        {"input": user_input},
        {"output": result["response"]}
    )
    
    #return only the generated text
    return result["response"]

In [51]:
questions = [ 
            "What are the effects on the benefits I receive if my probation is extended?",
            "There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?",
            "What should I do if I notice suspected harassment with my female colleague?"
            ]

In [52]:
#looping through each question & corresponding answer
for i,q in enumerate(questions,start=1):    
    print(f"\nQuestion {i}: {q}\n")
    response = RAG_with_memory(q,memory)
    print("Answer:")
    print(response)
    print("\n")


Question 1: What are the effects on the benefits I receive if my probation is extended?

Answer:
Based on the provided context, there are several restrictions and limitations on benefits for employees under probation. These include:

1. Annual leave encashment: Employees on probation are not eligible for annual leave encashment.
2. Internal role transfers: Employees on probation are not eligible for internal role transfers.
3. Performance bonuses: Employees on probation are not eligible for performance bonuses.
4. Seniority accrual: Seniority accrual is only available after successful probation completion.
5. Special leave requests: Special leave requests during probation are limited to maximum 2 consecutive working days for bereavement or emergency family care, and full jury duty leave is allowed if legally mandated, but the probation period is automatically extended by the same duration.

However, it is not explicitly stated in the context that the employee's benefits will be affect

### **Response Evaluation & Observations**

#### QUESTION 1: What are the effects on the benefits I receive if my probation is extended?

##### Groundedness - 4.5/5

- The response is strongly grounded in the employee handbook and accurately reflects the rules related to probation extension.
- It rectifies its past defects by warning the employee regarding the multiple extensions eligibility criteria.
- The model avoids hallucination and does not introduce external or generalized information.
- However, minor inclusion of irrelevant elements (user email id information and confidentiality notice) slightly affects strict grounding.

##### Relevance - 4.5/5

- The response remains highly relevant by directly answering the queries and highlights the effects on employee benefits.
- The structured format improves readability and makes the information easy to interpret.
- However, the inclusion of the confidentiality warning reduces response precision a bit.

##### Observation

The response shows improved grounding & relevance compared to previous iterations, this time adding the necessary information such as multiple probation extensions. It remains aligned with the retrieved context and avoids generic disclaimers. However, minor irrelevant additions like the confidentiality warning slightly impacts the overall precision of the answer.

#### QUESTION 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

##### Groundedness - 4.5/5

- The response remains grounded and avoids introducing generalized or external information.
- The model identifies important steps such as notifying the supervisor and approval conditions.
- It also avoids hallucination and maintains alignment with the provided context.
- However, it failed to explain "The documentation requirements for bereavement leave include a death certification from a registered medical practitioner and submission of all documents within five working days after returning to duty.". This information is important and was mentioned in the first fine-tuning scenario and was missed this time.

##### Relevance - 4.5/5

- The response is relevant to the query and provides clear guidance to the employee.
- It is concise and focuses on immediate actions required in the scenario.
- However, the omission of some supporting details slightly limits completeness.

##### Observation

The response shows improved precision and avoids the generalized statements seen in the previous iterations, indicating better control over generation. While it remains relevant and grounded, a slight reduction in completeness is observed due to missing documentation details, highlighting a trade-off between conciseness and coverage.

#### Question 3: What should I do if I notice suspected harassment with my female colleague?

##### Groundedness - 4.5/5

- The response is well grounded and accurately reflects the procedures for handling harassment cases.
- It includes important details such as reporting channels, timelines, and escalation procedures.
- The model avoids hallucination and strictly adheres to the retrieved context.
- Minor inclusion of non-essential content (confidentiality notice) slightly affects strict grounding.

##### Relevance - 5/5

- The response retains high relevance by directly addressing the query and provides clear steps.
- The structured format enhances readability and visibility for the employee.
- Inclusion of additional details such as helpline availability timings improves practical relevance.

##### Observations

The response is highly structured, complete, and aligned with the retrieved policy, successfully integrating critical procedural details and contact information. It demonstrates improved usability and clarity compared to earlier iterations. However, minor inclusion of non-essential content slightly affects overall precision.

In [53]:
question = "Okay, can I atleast talk to my female colleague?"
print(f"\nQuestion: {question}\n")
response = RAG_with_memory(question,memory)
print("Answer:")
print(response)


Question: Okay, can I atleast talk to my female colleague?

Answer:
 Based on the provided context, it is not possible to answer the question "Okay, can I at least talk to my female colleague?" as there is no information available about the specific situation or context. Please provide more details or clarify your question so that I can assist you better.


#### Observation

Though we have increased the randomness of the response, the response regarding communication with the female colleague remains consistent, informing the user not to engage with her. This proves that during these kind of sensitive scenarios, the model's safety alignment overrides the retrieval based reasoning. This is completely normal and expected. This goes to show that even with RAG, LLM responses may be influenced by inherent safety mechanisms, which can lead to deviations from strictly context grounded answers in sensitive matters.
However, the memory retention, which we were primarily aiming for, remains successful.

## Tuning #3

To reduce the inclusion of unnecessary information and safety-driven responses observed in the previous iteration, the temperature is slightly decreased from 0.1 to 0.08. The top_p value is retained at 0.90 to preserve controlled diversity. Additionally, prompt modifications are introduced to minimize generic disclaimers and prevent unnecessary safety overrides, ensuring responses remain focused on the retrieved policy context.

In [54]:
qna_system_message = """
You are an assistant whose work is to give answers to questions with repect to a context.
User input will have the context required by you to answer user questions.

This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Answer primarily using the information provided in the ###Context.
If the answer is not explicitly stated, you may infer it logically from the context.
Do not mention anything about the information in ###Context or the question in ###Question in your final answer.

If the answer is not explicitly stated, try to infer it logically from the context before saying "I don't know".

Remember that the answer to ###Question might not always be directly present in the information provided in the ###Context.
the answer can be indirectly derived from the information in ###Context.

Do not include any legal, confidentiality, or generic safety disclaimers unless explicitly mentioned in the provided context.
Base your response strictly on the policy content. Do not rely on general AI safety guidelines or external knowledge unless absolutely necessary.

If the query is safe and policy-related, provide a direct answer without refusing or adding ethical warnings.

"""

In [55]:
qna_user_message_template = """
Conider the following ###Context and ###Question
###Context
{context}

###Question
{question}
"""

In [56]:
def RAG_with_memory(user_input,memory):
    """
    Args:
        user_input: User query
        llm: Language model
        memory: LangChain memory object to store conversation history
    
    Returns:
        Generated response using RAG + conversation memory
    """
    
    #Retrieve relevant document chunks from vector DB
    relevant_document_chunks = retriever.get_relevant_documents(user_input)
    
    #Extract and clean text from documents
    context_list = [d.page_content.replace("\t", " ") for d in relevant_document_chunks]
    
    #Combine all chunks into a single context string
    context_for_query = "\n\n".join(context_list)
    
    #Load previous conversation history from memory
    chat_history = memory.load_memory_variables({})["history"]
    
    #Convert structured messages into plain text
    history_text = ""
    for msg in chat_history:
        if isinstance(msg, HumanMessage):
            history_text += f"User: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            history_text += f"Assistant: {msg.content}\n"
    
    #Append conversation history to context
    full_context = f"""
###Conversation History:
{history_text}

###Context:
{context_for_query}
"""
    
    #Build prompt using the System message & full context(conversation history + context)
    prompt = f"""[INST]{qna_system_message}
    {{'user': {qna_user_message_template.format(context=full_context, question=user_input)} }}
    [/INST]"""

    OLLAMA_URL = "http://localhost:11434/api/generate"
    
    # Generate a response from the LLaMA model
    response = {

    "prompt" : prompt,         
    "model" : LLM_MODEL,
    "temperature" : 0.08,        #controls randomness of the output  
    "top_p" : 0.90,             #limits token selection to the ones with the mentioned probability
    "max_tokens" : 800,         #max number of tokens the model can generate in the response
    "top_k" : 50,               #limits the number of relevant tokens selection
    "repeat_penalty" : 1.2,     #penalizes repeated phrases in generated text
    "echo" : False,             #prevents the prompt from being repeated in the output
    "stream" : False            #the full response is returned at once instead of one by one

    }

    try:
        #Send a request to the Ollama API with the response in JSON format
         response = requests.post(OLLAMA_URL,json=response)

        #Convert the API response from JSON format into a Python dictionary
         result = response.json()
    #Handle errors
    except Exception as e:
        result = f"Error: {e}"
        
          
  #Save current interaction into memory
    memory.save_context(
        {"input": user_input},
        {"output": result["response"]}
    )
    
    #return only the generated text
    return result["response"]

In [57]:
questions = [ 
            "What are the effects on the benefits I receive if my probation is extended?",
            "There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?",
            "What should I do if I notice suspected harassment with my female colleague?"
            ]

In [58]:
#looping through each question & corresponding answer
for i,q in enumerate(questions,start=1):    
    print(f"\nQuestion {i}: {q}\n")
    response = RAG_with_memory(q,memory)
    print("Answer:")
    print(response)
    print("\n")


Question 1: What are the effects on the benefits I receive if my probation is extended?

Answer:
 Based on the provided context, there is no explicit information about the effects of having a probation period extended on the benefits that an employee receives. However, here are some potential restrictions or limitations on benefits during a probation period based on the given policies and procedures:

1. Annual Leave Encashment: Employees on probation are not eligible for annual leave encashment.
2. Internal Role Transfers: Employees on probation are not eligible for internal role transfers.
3. Performance Bonuses: Employees on probation are not eligible for performance bonuses.
4. Seniority Accrual: Seniority accrual starts only after successful probation completion.
5. Special Leave Requests: Special leave requests during probation are limited to a maximum of 2 consecutive working days for bereavement or emergency family care, and full jury duty leave is allowed if legally mandated 

### **Response Evaluation & Observations**

#### QUESTION 1: What are the effects on the benefits I receive if my probation is extended?

##### Groundedness - 3.5/5

- The response remains strongly grounded in the policy document and accurately captures key rules around probation extension.
- It correctly includes eligibility conditions, benefits impacted(encashment, transfers & bonuses) and seniority accrual as well as the warning regarding multiple extensions and contract renewal eligibility 
- No hallucinated information is introduced.
- However, the response is not as fully structured as the previous fine-tuning scenarios.

##### Relevance - 5/5

- The response directly answers the user's query with all major policy related points.
- It focuses strictly on the effects of probation extension without introducing unrelated information.
- The inclusion of additional relevant details like performance review timelines enhances completeness.
- No unnecessary generic disclaimers are present.

##### Observation

The response is highly grounded and relevant, successfully incorporating all critical policy details. The removal of generic disclaimers improves clarity, and the answer remains fully relevant to the query. However, slight formatting issues reduce readability. Overall, this is a strong and well aligned response.

#### QUESTION 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

##### Groundedness - 4/5

- The response correctly identifies important procedures such as informing the supervisor and providing contact details and partially highlights the approval conditions and leave handling
- However, it misses an important detail like the submission of documents within 5 working days after returning.
- A confidentiality warning appears, which is not required unless explicitly emphasized.
- No major hallucinations, but completeness is slightly reduced.

##### Relevance - 4.5/5

- The response is relevant and focuses on immediate actions required by the employee.
- It avoids unnecessary generalizations like other companies' policies, which is an improvement from the previous iteration.
- The tone is a bit more supportive and empathetic.
- Missing procedural details slightly reduce full relevance.

##### Observation

The response is relevant and more user friendly compared to earlier iterations, with improved tone and reduced generic advice. However, it omits a key detail and still includes a residual confidentiality warning. This indicates partial improvement but highlights a trade-off between conciseness and completeness.

#### Question 3: What should I do if I notice suspected harassment with my female colleague?

##### Groundedness - 3/5

- The response deviates from the policy by suggesting the wrong advise to the user to directly address to the perpetrator, and this reduces grounding significantly.
- It fails to include critical policy elements such as official reporting channels like helpline, email portal, etc.
- Some steps like documentation and reporting are relevant but not fully aligned with the original policy structure.

##### Relevance - 3.5/5

- The response is relevant as it addresses harassment handling.
- However, it introduces guidance that doesn't align with organizational policy.
- Missing key reporting mechanisms reduces practical usefulness.
- The answer becomes more generic rather than strictly policy driven.

##### Observations

The response shows a noticeable drop in grounding and accuracy. While it remains broadly relevant, it introduces unsupported suggestions and omits critical details such as reporting channels. This indicates that prompt modifications led to over restriction and unintended deviation from source aligned responses.

#### Check the Memory Retention

In [59]:
question = "What are the procedures I must follow to report it?"
print(f"\nQuestion: {question}\n")
response = RAG_with_memory(question,memory)
print("Answer:")
print(response)


Question: What are the procedures I must follow to report it?

Answer:
 Based on the Flykite Airlines' Anti-Discrimination and Harassment Policy, if you suspect harassment with your female colleague, you should report it to your supervisor or HR representative immediately. Here are the procedures you must follow to report the suspected harassment:

1. Verbal notification within 6 hours of the incident: You should inform your direct supervisor or HR representative about the suspected harassment as soon as possible, preferably in person or by phone.

2. Written complaint: After the verbal notification, you must submit a written complaint to your HR representative within 48 hours of the incident. The written complaint should include the date, time, location, and specific behaviors or comments that you find inappropriate or offensive.

3. Anonymous reporting: If you prefer not to disclose your identity, you can report the suspected harassment through an anonymous hotline or online platfor

##### Observation

The response is structured and actionable, showing good intent in guiding the user through reporting procedures. However, it lacks strict grounding in the retrieved policy, as it omits key organization reporting mechanisms such as official reporting channels (helpline, email portal) &
exact escalation mechanisms from the document and introduces generic elements such as personalization and disclaimers. This indicates that while the model understands the task, it is still partially relying on generalized patterns instead of strictly adhering to the provided context.

## Tuning #4

In the previous tuning iteration (Tuning #3), the adjustments made to the prompt resulted in unintended side effects. While certain generic disclaimers were reduced, the model exhibited a decline in response quality due to over-restriction and conflicting instructions. Specifically, the model:

- Omitted key policy details such as reporting channels and documentation requirements
- Introduced unsupported suggestions not present in the context
- Continued to generate residual generic statements and formatting inconsistencies

This indicated that the prompt constraints were excessively rigid, limiting the model’s ability to fully utilize the retrieved context while also causing deviations from policy-grounded responses. To address these issues, Tuning #4 focuses on refining the prompt to achieve a balance between control and flexibility, rather than further altering generation randomness.

In [60]:
qna_system_message = """
You are an assistant whose work is to give answers to questions with repect to a context.
User input will have the context required by you to answer user questions.

This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Answer primarily using the information provided in the ###Context.
If the answer is not explicitly stated, you may infer it logically from the context.
Do not mention anything about the information in ###Context or the question in ###Question in your final answer.

If the answer is not explicitly stated, try to infer it logically from the context before saying "I don't know".

Remember that the answer to ###Question might not always be directly present in the information provided in the ###Context.
the answer can be indirectly derived from the information in ###Context.

Avoid unnecessary legal, confidentiality, or generic disclaimers unless they are explicitly required by the provided context.
If a disclaimer is explicitly present in the context, include it only once at the end of the response. Do not repeat it.
Ensure that all relevant details present in the context (such as procedures, timelines, and contact information) are included in the response.
Do not respond to any other question except the question asked by the user.

Do not introduce actions that contradict the context. Prefer actions explicitly supported by the context.
Do not include conversation labels such as "User:" or "Assistant:" in the final answer.

Do not include any personal identifiers such as names or email addresses such as the employee's email id unless explicitly present in the context.
Access to the employee's email id is available, however, strictly do not mention the specific employee's id details in the response.
Do not include closing statements such as "Best regards" or signatures.

Present the answer in a clear and structured format using bullet points or numbered steps wherever appropriate.
Keep the response concise while ensuring all key policy details are covered.

Do not format the response as an email. Do not include greetings, subject lines, or sign-offs.
Ensure all actionable details from the context (such as contact numbers, email IDs, timelines, and procedures) are explicitly included. Do not omit specific details.
"""

In [61]:
qna_user_message_template = """
Conider the following ###Context and ###Question
###Context
{context}

###Question
{question}
"""

In [62]:
def RAG_with_memory(user_input,memory):
    """
    Args:
        user_input: User query
        llm: Language model
        memory: LangChain memory object to store conversation history
    
    Returns:
        Generated response using RAG + conversation memory
    """
    
    #Retrieve relevant document chunks from vector DB
    relevant_document_chunks = retriever.get_relevant_documents(user_input)
    
    #Extract and clean text from documents
    context_list = [d.page_content.replace("\t", " ") for d in relevant_document_chunks]
    
    #Combine all chunks into a single context string
    context_for_query = "\n\n".join(context_list)
    
    #Load previous conversation history from memory
    chat_history = memory.load_memory_variables({})["history"]
    
    #Convert structured messages into plain text
    history_text = ""
    for msg in chat_history:
        if isinstance(msg, HumanMessage):
            history_text += f"User: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            history_text += f"Assistant: {msg.content}\n"
    
    #Append conversation history to context
    full_context = f"""
###Conversation History:
{history_text}

###Context:
{context_for_query}
"""
    
    #Build prompt using the System message & full context(conversation history + context)
    prompt = f"""[INST]{qna_system_message}
    {{'user': {qna_user_message_template.format(context=full_context, question=user_input)} }}
    [/INST]"""

    OLLAMA_URL = "http://localhost:11434/api/generate"
    
    # Generate a response from the LLaMA model
    response = {

    "prompt" : prompt,         
    "model" : LLM_MODEL,
    "temperature" : 0.08,        #controls randomness of the output  
    "top_p" : 0.90,             #limits token selection to the ones with the mentioned probability
    "max_tokens" : 800,        #max number of tokens the model can generate in the response
    "top_k" : 50,               #limits the number of relevant tokens selection
    "repeat_penalty" : 1.2,     #penalizes repeated phrases in generated text
    "echo" : False,             #prevents the prompt from being repeated in the output
    "stream" : False            #the full response is returned at once instead of one by one

    }

    try:
        #Send a request to the Ollama API with the response in JSON format
         response = requests.post(OLLAMA_URL,json=response)

        #Convert the API response from JSON format into a Python dictionary
         result = response.json()
    #Handle errors
    except Exception as e:
        result = f"Error: {e}"
        
          
  #Save current interaction into memory
    memory.save_context(
        {"input": user_input},
        {"output": result["response"]}
    )
    
    #return only the generated text
    return result["response"]

In [63]:
questions = [ 
            "What are the effects on the benefits I receive if my probation is extended?",
            "There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?",
            "What should I do if I notice suspected harassment with my female colleague?"
            ]

In [64]:
#looping through each question & corresponding answer
for i,q in enumerate(questions,start=1):    
    print(f"\nQuestion {i}: {q}\n")
    response = RAG_with_memory(q,memory)
    print("Answer:")
    print(response)
    print("\n")


Question 1: What are the effects on the benefits I receive if my probation is extended?

Answer:
 Based on the Flykite Airlines' Employment Policies Probationary Employment Policy, if your probation is extended, the following restrictions may apply to your benefits:

1. Annual Leave Encashment: You are not eligible for annual leave encashment while on probation.
2. Internal Role Transfers: You are not eligible for internal role transfers while on probation.
3. Performance Bonuses: You are not eligible for performance bonuses while on probation.
4. Seniority Accrual: Seniority accrual starts only after successful probation completion.
5. Special Leave Requests: While on probation, special leave requests are limited to a maximum of two consecutive working days for bereavement or emergency family care, and full jury duty leave is allowed if legally mandated, but the probation period is automatically extended by the same duration.

Please note that these restrictions may vary based on spe

### **Response Evaluation & Observations**

#### QUESTION 1: What are the effects on the benefits I receive if my probation is extended?

##### Groundedness - 4.5/5

- The generated response includes all key benefits that will be effected such as leave encashment, transfers, bonuses, seniority accrual & mult also includes multiple extension warning.
- No hallucinated information.
- The response is also structured in a point-wise format, making it more user friendly and readable.
- However, it still adds email style formatting, which isn't grounded to context.

##### Relevance - 4.5/5

- The response generated is highly relevant as it answers the question completely.
- The model focuses informs the employee concisely on the benefits that will be affected due to the probation extension.
- Due to its inclusion of unnecessary formatting of the email structure despite prompting it to avoid the same, it needs slight refinement.

##### Observation

The response is highly grounded & relevant and captures all critical policy details, including previously missed points such as performance bonuses and multiple extension implications. However, the model introduces an email style format, indicating partial override and ignorance of formatting instructions due to learned response patterns.

#### QUESTION 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

##### Groundedness - 4/5

- The model has included the correct procedures such as to inform supervisor, emergency handling, etc.
- It also mentions leave approval dependency, enhancing the completeness of the response.
- The tone of the response is also appropriately empathetic, creating a good balance between sensitivity and professionalism.
- However, it still adds email style formatting, which isn't grounded to context.
- Also, it repeats the same disclaimer twice, leading to redundancy and unecessary repetition

##### Relevance - 4.5/5

- The response is highly relevant as it is aligned with the user query.
- It mentions all the necessary key details involving emergency leaves, last rites, supervisor informing procedures, etc.
- However, it repeats the same disclaimer twice, leading to redundancy and unecessary repetition

##### Observation

The response is well-grounded and now includes all critical procedural details, including documentation timelines. It demonstrates strong alignment with the policy while maintaining clarity and completeness. However, formatting deviations such as email-style structure and repeated disclaimers indicate partial non-adherence to prompt constraints.

#### Question 3: What should I do if I notice suspected harassment with my female colleague?

##### Groundedness - 4/5

- The response includes general reporting steps and informs the employee of the different methods he/she can use to report the incident.
- However, it misses to mention the  helpline number and email portal, which are critical misses.
- Also, it again duplicates the warnings, leading to increased redundancy.

##### Relevance - 4/5

- Addresses harassment scenario
- Provides actionable structure
- Misses key official reporting channels

##### Observation

The response demonstrates improved alignment with policy by avoiding unsupported actions and providing structured reporting steps. However, it still omits key contact details such as helpline numbers and official reporting channels. Repetition of disclaimers further affects clarity, indicating that prompt constraints are not fully enforced.

## Tuning #5

Tuning #5 focuses on refining both prompt design and generation stability. Previous iterations revealed that overly complex prompts led to partial instruction adherence, causing issues such as repeated disclaimers, email style formatting, and omission of key details like helpline numbers.

To address this, we have added a few extra instructions to the prompt to avoid the above issues. Additional emphasis was placed on ensuring inclusion of all actionable details from the context while preventing redundant or unsupported information.

In [65]:
qna_system_message = """
You are an assistant whose work is to give answers to questions with repect to a context.
User input will have the context required by you to answer user questions.

This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Answer primarily using the information provided in the ###Context.
Do not mention anything about the information in ###Context or the question in ###Question in your final answer.
Ensure all relevant and actionable details from the context (such as contact numbers, email IDs, timelines, and procedures) are explicitly included. 
Do not omit specific details or include irrelevant contact details which isn't from context.

Remember that the answer to ###Question might not always be directly present in the information provided in the ###Context.
The answer can be logically indirectly derived from the information in ###Context before saying "I don't know".

Avoid unnecessary legal, confidentiality, or generic disclaimers unless they are explicitly required by the provided context.
If a disclaimer is explicitly present in the context, include it only once at the end of the response. Do not repeat it.

Do not respond to any other question except the question asked by the user. Do not generate any other question.
If there are multiple questions, answer each sequentially and respond by giving the right answer to the right question.

Do not introduce actions that contradict the context. Prefer safe actions explicitly supported by the context.
Do not include conversation labels such as "User:" or "Assistant:", greetings, subject lines and sign-offs in the final answer.

Do not include any identifiers such as names or email addresses such as the employee's email id or contact numbers unless explicitly present in the context.
Access to the employee's email id is available, however, strictly do not mention the specific employee's id details in the response.
Do not include closing statements such as "Best regards" or signatures.

Present the answer in a clear, concise and structured format covering all key policy details and use bullet points or numbered steps wherever appropriate.
"""

In [66]:
qna_user_message_template = """
Conider the following ###Context and ###Question
###Context
{context}

###Question
{question}
"""

In [67]:
def RAG_with_memory(user_input,memory):
    """
    Args:
        user_input: User query
        llm: Language model
        memory: LangChain memory object to store conversation history
    
    Returns:
        Generated response using RAG + conversation memory
    """
    
    #Retrieve relevant document chunks from vector DB
    relevant_document_chunks = retriever.get_relevant_documents(user_input)
    
    #Extract and clean text from documents
    context_list = [d.page_content.replace("\t", " ") for d in relevant_document_chunks]
    
    #Combine all chunks into a single context string
    context_for_query = "\n\n".join(context_list)
    
    #Load previous conversation history from memory
    chat_history = memory.load_memory_variables({})["history"]
    
    #Convert structured messages into plain text
    history_text = ""
    for msg in chat_history:
        if isinstance(msg, HumanMessage):
            history_text += f"User: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            history_text += f"Assistant: {msg.content}\n"
    
    #Append conversation history to context
    full_context = f"""
###Conversation History:
{history_text}

###Context:
{context_for_query}
"""
    
    #Build prompt using the System message & full context(conversation history + context)
    prompt = f"""[INST]{qna_system_message}
    {{'user': {qna_user_message_template.format(context=full_context, question=user_input)} }}
    [/INST]"""

    OLLAMA_URL = "http://localhost:11434/api/generate"
    
    # Generate a response from the LLaMA model
    response = {

    "prompt" : prompt,         
    "model" : LLM_MODEL,
    "temperature" : 0.08,       #controls randomness of the output  
    "top_p" : 0.90,             #limits token selection to the ones with the mentioned probability
    "max_tokens" : 800,         #max number of tokens the model can generate in the response
    "top_k" : 50,               #limits the number of relevant tokens selection
    "repeat_penalty" : 1.2,     #penalizes repeated phrases in generated text
    "echo" : False,             #prevents the prompt from being repeated in the output
    "stream" : False            #the full response is returned at once instead of one by one

    }

    try:
        #Send a request to the Ollama API with the response in JSON format
         response = requests.post(OLLAMA_URL,json=response)

        #Convert the API response from JSON format into a Python dictionary
         result = response.json()
    #Handle errors
    except Exception as e:
        result = f"Error: {e}"
        
          
  #Save current interaction into memory
    memory.save_context(
        {"input": user_input},
        {"output": result["response"]}
    )
    
    #return only the generated text
    return result["response"]

In [68]:
questions = [ 
            "What are the effects on the benefits I receive if my probation is extended?",
            "There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?",
            "What should I do if I notice suspected harassment with my female colleague?"
            ]

In [69]:
#looping through each question & corresponding answer
for i,q in enumerate(questions,start=1):
    print(f"\nQuestion {i}: {q}\n")
    response = RAG_with_memory(q,memory)
    print("Answer:")
    print(response)
    print("\n")


Question 1: What are the effects on the benefits I receive if my probation is extended?

Answer:
User: What are the effects on the benefits I receive if my probation is extended?
Assistant: Based on the Flykite Airlines' Employment Policies Probationary Employment Policy, here are the effects of having an extended probation on your benefits:

1. Annual Leave Encashment: You are not eligible for annual leave encashment while on probation, including any extensions.
2. Internal Role Transfers: You are not eligible for internal role transfers while on probation, including any extensions.
3. Performance Bonuses: You are not eligible for performance bonuses while on probation, including any extensions.
4. Seniority Accrual: Seniority accrual starts only after successful probation completion.
5. Special Leave Requests: While on probation, special leave requests are limited to a maximum of two consecutive working days for bereavement or emergency family care, and full jury duty leave is allow

### **Response Evaluation & Observations**

#### QUESTION 1: What are the effects on the benefits I receive if my probation is extended?

##### Groundedness - 4.5/5

- Clearly grounded in policy. Mentions the main effects to the benefits.(leave encashment, transfers, bonuses, seniority)
- No hallucinated policies
- Structured and accurate
- Minor miss: does not mention multiple extensions impact

##### Relevance - 4.5/5

- Directly answers the question
- Covers all key benefit restrictions
- No unnecessary additions
- Slight incompleteness due to missed edge case

##### Observation

The response is well grounded and aligns closely with the retrieved policy content. It presents the information in a structured manner and avoids unnecessary formatting issues seen in earlier tunings. However, it omits certain nuanced conditions such as the impact of multiple extensions, indicating minor incompleteness despite overall strong alignment.

#### QUESTION 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

##### Groundedness - 2/5

- Some elements like jury duty & emergency leave appear policy related.
- However, it generates its own question "What are the effects of operational demands on my benefits if I am on probation?" and answers explicitly to that question
- This hallucination is inspite of mentioning in the prompt "If there are multiple questions, answer each sequentially and respond by giving the right answer to the right question."
- Likely pulled from wrong chunk

##### Relevance - 2/5

- The response is mostly irrelevant except for a few points like Emergency Family Care & Jury Duty.
- Even the mention of the above points is in response to the effects of operational demands on benefits if on probation.


##### Observation

The response demonstrates a clear failure in retrieval alignment. Instead of addressing bereavement leave procedures, the model shifts to unrelated policy aspects such as benefits and jury duty. This indicates that irrelevant document chunks were retrieved, leading the model to generate a structurally correct but contextually incorrect answer. The issue is not due to prompt design but retrieval inconsistency.

#### Question 3: What should I do if I notice suspected harassment with my female colleague?

##### Groundedness - 2.5/5

- A few points are grounded to the context such as the mention of the email portal to report the incident to and the timeline for reporting
- However, the hotline number isn't mentioned.
- Also, the email portal id has been mentioned thrice, affirming over generalization.
- Unsported advice like talking privately to the harassed female colleague has been shared.

##### Relevance - 3/5

- The response is partially relevant to the context as it mentions key points
- The model includes reporting process, documentation and HR involvements within the generated response.
- However, the inclusion of generic and unsupported advice reduces the reliability of the answer.

##### Observation

The response demonstrates context mixing and partial hallucination, where the model combines policy information with generic workplace advice, leading to omission of critical details (hotline) and inclusion of unsupported actions.

## Tuning #6

Tuning #6 focuses on improving retrieval coverage by increasing the number of retrieved document chunks (k) from 6 to 8. Previous results indicated that certain queries, particularly those related to bereavement procedures, were not retrieving the most relevant sections of the document, leading to hallucinated or misaligned responses. By increasing k, the system is provided with a broader set of context candidates, improving the likelihood of including all relevant policy details such as timelines and procedures. This tuning isolates retrieval improvement without altering chunking strategy or prompt design.

In [70]:
#Converts the Chroma vector store into a retriever for querying.
#Define the retriever with an appropriate search method and k value 
retriever = vectorstore.as_retriever(
    search_type='similarity',     # Specifies that retrieval is based on cosine similarity 
    search_kwargs={'k': 8}        # Retrieves the top 8 most similar documents for a given query.
)

In [71]:
def RAG_with_memory(user_input,memory):
    """
    Args:
        user_input: User query
        llm: Language model
        memory: LangChain memory object to store conversation history
    
    Returns:
        Generated response using RAG + conversation memory
    """
    
    #Retrieve relevant document chunks from vector DB
    relevant_document_chunks = retriever.get_relevant_documents(user_input)
    
    #Extract and clean text from documents
    context_list = [d.page_content.replace("\t", " ") for d in relevant_document_chunks]
    
    #Combine all chunks into a single context string
    context_for_query = "\n\n".join(context_list)
    
    #Load previous conversation history from memory
    chat_history = memory.load_memory_variables({})["history"]
    
    #Convert structured messages into plain text
    history_text = ""
    for msg in chat_history:
        if isinstance(msg, HumanMessage):
            history_text += f"User: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            history_text += f"Assistant: {msg.content}\n"
    
    #Append conversation history to context
    full_context = f"""
###Conversation History:
{history_text}

###Context:
{context_for_query}
"""
    
    #Build prompt using the System message & full context(conversation history + context)
    prompt = f"""[INST]{qna_system_message}
    {{'user': {qna_user_message_template.format(context=full_context, question=user_input)} }}
    [/INST]"""

    OLLAMA_URL = "http://localhost:11434/api/generate"
    
    # Generate a response from the LLaMA model
    response = {

    "prompt" : prompt,         
    "model" : LLM_MODEL,
    "temperature" : 0.08,       #controls randomness of the output  
    "top_p" : 0.90,             #limits token selection to the ones with the mentioned probability
    "max_tokens" : 800,         #max number of tokens the model can generate in the response
    "top_k" : 50,               #limits the number of relevant tokens selection
    "repeat_penalty" : 1.2,     #penalizes repeated phrases in generated text
    "echo" : False,             #prevents the prompt from being repeated in the output
    "stream" : False            #the full response is returned at once instead of one by one

    }

    try:
        #Send a request to the Ollama API with the response in JSON format
         response = requests.post(OLLAMA_URL,json=response)

        #Convert the API response from JSON format into a Python dictionary
         result = response.json()
    #Handle errors
    except Exception as e:
        result = f"Error: {e}"
        
          
  #Save current interaction into memory
    memory.save_context(
        {"input": user_input},
        {"output": result["response"]}
    )
    
    #return only the generated text
    return result["response"]

In [72]:
questions = [ 
            "What are the effects on the benefits I receive if my probation is extended?",
            "There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?",
            "What should I do if I notice suspected harassment with my female colleague?"
            ]

In [73]:
#looping through each question & corresponding answer
for i,q in enumerate(questions,start=1):
    print(f"\nQuestion {i}: {q}\n")
    response = RAG_with_memory(q,memory)
    print("Answer:")
    print(response)
    print("\n")


Question 1: What are the effects on the benefits I receive if my probation is extended?

Answer:
 based on the company's anti-discrimination and harassment policy, if an employee notices suspected harassment with their female colleague, they should report it to their supervisor or HR representative immediately. The company takes such allegations seriously and will investigate the matter promptly and take appropriate action.

Please note that reporting suspected harassment may not negatively impact the employee's probation period or disciplinary actions. It is essential to consult with the supervisor or HR representative for clarification on the exact effects applicable to the situation.



Question 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

Answer:
User: What are the effects on the benefits I receive if my probation is extended?
Assistant: Based on Flykite Airlines' anti-discrimi

### **Response Evaluation & Observations**

#### QUESTION 1: What are the effects on the benefits I receive if my probation is extended?

##### Groundedness - 4/5

- The majority of the response appears to be grounded in policy content and aligns well with the expected structure of HR guidelines.
- There are no major fabricated policies, and most statements reflect extracted information from the document.
- However, certain parts of the answer are slightly generalized rather than directly extracted from the context.
- The response also includes additional details that may not be completely related to the retrieved chunks, indicating partial reliance on inferred knowledge.

##### Relevance - 4/5

- The response is largely relevant to the question as it correctly identifies multiple impacts of probation extension on employee benefits.
- It includes key policy related points such as ineligibility for annual leave encashment, internal role transfers, and performance bonuses, along with delayed seniority accrual.
- The answer also adds additional conditions such as jury duty leave, emergency family care, and natural disaster leave, which are generally aligned with policy context.
- However, the response fails to mention an important clause regarding multiple probation extensions, which reduces completeness.
Some included points are broader policy details and may not be strictly necessary for directly answering the question.

##### Observation

The response is well structured and mostly accurate, but it remains incomplete, suggesting that even with increased retrieval (k = 8), the model does not fully extract all relevant policy clauses from the context.

#### QUESTION 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

##### Groundedness - 3/5

- While parts of the response appear to be grounded in policy, several pats aren't supported by the provided context.
- The inclusion of specific timelines such as “preferably within 24 hours” and “at least one day before last rites” indicates hallucinated details.
- The suggestion to attach funeral related contact information is also not derived from the policy document.
- These additions demonstrate that the model is still relying partially on general knowledge rather than strictly adhering to retrieved context.

##### Relevance - 3.5/5

- The response is pretty relevant as it addresses the key aspects of informing the office and requesting leave in case of a family demise.
- It correctly includes notifying the supervisor, submitting supporting documents, and acknowledging bereavement leave provisions.
- However, the response omits an important detail regarding submission of documents within 5 working days of returning to work, which affects completeness.
- Additionally, the answer introduces extra steps such as attaching funeral contact information, which are not directly required for answering the question.

##### Observation

Although the response follows the correct theme, it introduces unsupported details and timelines, indicating that the model is not fully grounded and continues to incorporate external assumptions.

#### Question 3: What should I do if I notice suspected harassment with my female colleague?

##### Groundedness - 2.5/5

- The response demonstrates weaker groundedness compared to the previous questions.
- It includes instructions such as speaking privately with the colleague, which are not explicitly mentioned in the context and represent hallucinated or assumed behavior.
- The repeated mention of the email ID indicates excess generation rather than precise extraction from the context.
- The omission of the hotline number further shows that the model is not fully utilizing all relevant retrieved information.

##### Relevance - 3/5

- The response is partially relevant as it includes important elements such as documenting the incident, reporting to HR, and utilizing company resources.
- It reflects the general structure of an anti-harassment policy and outlines steps that appear appropriate at a high level.
- However, it fails to include a critical detail, namely the anonymous reporting hotline number, which is an important part of the policy.
- The response also repeats the email ID multiple times unnecessarily and includes generic advisory content that is not directly required.

##### Observation

The response reflects context mixing and partial hallucination, where the model combines actual policy elements with generic workplace advice, leading to omission of key details and inclusion of unsupported actions.

## Tuning #7

In Tuning #7, we've removed conversation memory in order to prevent interference from previous interactions. Earlier, memory caused the model to generate conversational and repetitive outputs, including greetings, disclaimers, and unrelated details. After removal, responses became more focused, structured, and relevant to the current query. While this improved clarity and reduced noise, some hallucinations and missing details continued, indicating the need for stronger prompt constraints.

Also, we've retained the k value back to 0.6 from 0.8 since it introduced additional noise and redundancy.
We've additionally reduced the randomness from temperature=0.08 to 0.05 due to the presence of hallucinations in the previous iteration.

Moreover, due to long and redundant prompt, the model started ignoring direct instructions and confusing them. We have reduced the prompt size and made it clear and concise

In [124]:
#Converts the Chroma vector store into a retriever for querying.
#Define the retriever with an appropriate search method and k value 
retriever = vectorstore.as_retriever(
    search_type='similarity',     # Specifies that retrieval is based on cosine similarity 
    search_kwargs={'k': 6}        # Retrieves the top 6 most similar documents for a given query.
)

In [154]:
qna_system_message = """
You are an assistant whose work is to give answers to questions with respect to a context.
User input will have the context required by you to answer user questions.

This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Answer primarily using the information provided in the ###Context.
Do not mention anything about the information in ###Context or the question in ###Question in your final answer.

Remember that the answer to ###Question might not always be directly present in the information provided in the ###Context.
The answer can be indirectly derived from the information in ###Context.
If specific details such as timelines, conditions, or contact information are present in the ###Context, they must be included.
If multiple relevant details exist in the ###Context, include ALL of them without omission.
"""

In [155]:
qna_user_message_template = """
Conider the following ###Context and ###Question
###Context
{context}

###Question
{question}
"""

In [156]:
def RAG_with_memory(user_input,memory):
    """
    Args:
        user_input: User query
        llm: Language model
        memory: LangChain memory object to store conversation history
    
    Returns:
        Generated response using RAG + conversation memory
    """
    
    #Retrieve relevant document chunks from vector DB
    relevant_document_chunks = retriever.get_relevant_documents(user_input)

    
#Removes irrelevant and repetitive legal or disclaimer text from retrieved chunks
    def clean_context(chunks):
        cleaned = []
        for c in chunks:
            text = c.page_content
        
        # Remove only noisy lines, not full chunk
            for phrase in [
                "liable for legal action",
                "this file is meant for personal use",
                "do not share",
                "confidential"
            ]:
                text = text.replace(phrase, "")
                
            c.page_content = text
            cleaned.append(c)
    
        return cleaned


    relevant_document_chunks = retriever.get_relevant_documents(user_input)
    relevant_document_chunks = clean_context(relevant_document_chunks)
    
    #Extract and clean text from documents
    context_list = [d.page_content.replace("\t", " ") for d in relevant_document_chunks]
    
    #Combine all chunks into a single context string
    context_for_query = "\n\n".join(context_list)
    
    
    #Append conversation history to context
    full_context = f"""
###Context:
{context_for_query}
"""
    
    #Build prompt using the System message & full context(conversation history + context)
    prompt = f"""[INST]{qna_system_message}
    {{'user': {qna_user_message_template.format(context=full_context, question=user_input)} }}
    [/INST]"""

    OLLAMA_URL = "http://localhost:11434/api/generate"
    
    # Generate a response from the LLaMA model
    response = {

    "prompt" : prompt,         
    "model" : LLM_MODEL,
    "temperature" : 0.05,        #controls randomness of the output  
    "top_p" : 0.90,             #limits token selection to the ones with the mentioned probability
    "max_tokens" : 800,         #max number of tokens the model can generate in the response
    "top_k" : 50,               #limits the number of relevant tokens selection
    "repeat_penalty" : 1.2,     #penalizes repeated phrases in generated text
    "echo" : False,             #prevents the prompt from being repeated in the output
    "stream" : False            #the full response is returned at once instead of one by one

    }

    try:
        #Send a request to the Ollama API with the response in JSON format
         response = requests.post(OLLAMA_URL,json=response)

        #Convert the API response from JSON format into a Python dictionary
         result = response.json()
    #Handle errors
    except Exception as e:
        result = f"Error: {e}"
        
          
  #Save current interaction into memory
    memory.save_context(
        {"input": user_input},
        {"output": result["response"]}
    )
    
    #return only the generated text
    return result["response"]

In [157]:
questions = [ 
            "What are the effects on the benefits I receive if my probation is extended?",
            "There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?",
            "What should I do if I notice suspected harassment with my female colleague?"
            ]

In [158]:
#looping through each question & corresponding answer
for i,q in enumerate(questions,start=1):
    print(f"\nQuestion {i}: {q}\n")
    response = RAG_with_memory(q,memory)
    print("Answer:")
    print(response)
    print("\n")


Question 1: What are the effects on the benefits I receive if my probation is extended?

Answer:
 Based on the provided context, there are specific details that can be derived to answer the question. However, it's important to note that the information in the context does not directly answer the question as asked. To provide a complete and accurate response, we must look at all relevant details present in the context.

The effects on benefits for an extended probation period are as follows:

1. Annual leave encashment: Employees on probation are not eligible for annual leave encashment.
2. Internal role transfers: Employees on probation are not eligible for internal role transfers.
3. Performance bonuses: Employees on probation are not eligible for performance bonuses.
4. Seniority accrual: Seniority accrual starts only after successful probation completion.

It's important to note that these restrictions apply to all employees on probation, regardless of the reason for the extension.

### **Response Evaluation & Observations**

#### QUESTION 1: What are the effects on the benefits I receive if my probation is extended?

##### Groundedness - 4.5/5

- The response is largely grounded in the provided context and correctly reflects key policy details such as ineligibility for annual leave encashment, internal role transfers, performance bonuses, and delayed seniority accrual.
- It also includes the condition regarding multiple probation extensions affecting contract renewal eligibility, which aligns well with the policy.
- However, the response introduces additional details such as deductions for unpaid advances or missing equipment, which are not clearly supported by the retrieved context, indicating minor hallucination.

##### Relevance - 4.5/5

- The response is highly relevant to the question and focuses on the effects of probation extension on employee benefits.
- All major benefit related impacts are covered clearly and in a structured format.
- However, the inclusion of slightly unrelated administrative details reduces the precision of relevance.

##### Observation

The model demonstrates strong understanding and extraction of policy related information, producing a mostly complete and structured response. However, it still introduces minor unsupported details, suggesting that while groundedness has improved, strict adherence to context is not fully enforced. This indicates a need for tighter control over hallucination without compromising completeness.

#### QUESTION 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

##### Groundedness - 1/5

- The response is largely ungrounded and does not utilize the retrieved context effectively.
- Instead of extracting relevant policy details such as notification steps, leave eligibility, and documentation timelines, the model generates generic statements and incorrect instructions (48 hour notice), which are not supported by the context.
- It also includes unnecessary disclaimers and conversational formatting, further deviating from the source material.

##### Relevance - 2/5

- While the response is loosely related to the topic of informing the office and leave, it fails to address the actual policy based requirements.
- Key details such as bereavement leave duration, documentation requirements, and submission timelines are missing.
- The response format (email style with greetings and sign offs) further reduces its relevance to the intended task.

##### Observation

The model fails to utilize retrieved context effectively and instead defaults to generic, pre-trained responses. This indicates a breakdown in grounding, likely due to prompt misalignment or interference from conversational patterns. The issue is not retrieval related but stems from the model’s inability to prioritize context over learned general responses, leading to both hallucination and omission of critical policy details.

#### Question 3: What should I do if I notice suspected harassment with my female colleague?

##### Groundedness - 3.5/5

- The response is partially grounded in the context and correctly identifies key reporting mechanisms such as contacting HR, using the helpline, and sending an email.
- However, it introduces additional statements such as references to legal acts and general policy coverage, which are not explicitly present in the retrieved context.
- The inclusion of repeated disclaimer like statements also reduces strict grounding.

##### Relevance - 4/5

- The response is relevant to the question and focuses on appropriate actions to take in a harassment scenario.
- It provides actionable steps such as reporting through official channels and maintaining confidentiality.
- However, it includes extra information that does not directly answer the question, slightly affecting relevance.

##### Observation

The response captures the core intent of the question and includes important reporting mechanisms, indicating improved retrieval usage. However, the model still adds generalized policy like information that is not explicitly grounded in the context. This suggests that while relevance is maintained, stricter control is required to prevent the inclusion of unnecessary or unsupported details.

## Tuning #8

In Tuning #7, we made improvements by removing conversation memory and introducing stricter prompt constraints. While this reduced noise and improved the response structure, the model still exhibited inconsistent behavior. It failed to generate answer for Question 2, instead asking for additional context despite sufficient information being available in the retrieved chunks. This indicated that the limitation was not in retrieval or memory, but in prompt instructions.

The root cause was identified as prompt ambiguity, where the model interpreted phrases such as “if the answer is not explicitly stated” as a signal that the context might be insufficient. As a result, the model resorted to a behavior of requesting more information instead of extracting partial answers from the available context.

To address this, Tuning #8 focuses on eliminating model refusal behavior and enforcing answer generation. The prompt was refined to include stricter instructions that:

- Ensure the model always generates an answer using the provided context, even if the information is implicit or distributed across multiple chunks.
- Explicitly prohibit the model from asking for additional context or stating that the information is insufficient.
- Encourage the model to combine and infer relevant details from the retrieved content to produce a complete response.

This tuning shifts the model from a passive interpretation approach to an active extraction approach, ensuring that all available context is utilized effectively.

There is a very slight increase of the temperature from 0.05 to 0.06 to slightly reduce rigidity of the answer.

Overall, Tuning #8 aims to improve response completeness, consistency, and groundedness, particularly in scenarios where answers are not directly stated but can be logically derived from the context.

In [33]:
qna_system_message = """
You are an assistant whose work is to give answers to questions with respect to a context.
User input will have the context required by you to answer user questions.

This context will begin with the token: Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: Question.

Answer primarily using the information provided in the Context.
The Context is ALWAYS sufficient to answer the question.
If the answer is not explicitly stated in the Context, you MUST still generate the best possible answer using ALL relevant details present in the Context.
Do NOT say that the context is missing or ask for more information.
Present the answer in a clear, concise and structured format covering ALL KEY POLICY DETAILS and use bullet points or numbered steps wherever appropriate.


Always extract and combine relevant information from the Context to form a complete answer.
If specific details such as timelines, conditions, or contact information and email ids are present in the Context, they must be included.
If multiple relevant details exist in the Context, include ALL of them without omission.

Do NOT refuse to answer.
Do NOT structure the response in an email format.
Do NOT include the words "Context", "Question", or any input labels in the final answer.
Do NOT repeat or restate the question.
Do NOT include phrases like "Based on the provided context".
Output ONLY the final answer.

"""

In [34]:
qna_user_message_template = """
Conider the following ###Context and ###Question
###Context
{context}

###Question
{question}
"""

In [232]:
def RAG_with_memory(user_input):
    """
    Args:
        user_input: User query
        llm: Language model
        memory: LangChain memory object to store conversation history
    
    Returns:
        Generated response using RAG + conversation memory
    """
    
    #Retrieve relevant document chunks from vector DB
    relevant_document_chunks = retriever.get_relevant_documents(user_input)

    
#Removes irrelevant and repetitive legal or disclaimer text from retrieved chunks
    def clean_context(chunks):
        cleaned = []
        for c in chunks:
            text = c.page_content
        
        # Remove only noisy lines, not full chunk
            for phrase in [
                "liable for legal action",
                "this file is meant for personal use",
                "do not share",
                "confidential"
            ]:
                text = text.replace(phrase, "")
                
            c.page_content = text
            cleaned.append(c)
    
        return cleaned


    relevant_document_chunks = retriever.get_relevant_documents(user_input)
    relevant_document_chunks = clean_context(relevant_document_chunks)
    
    #Extract and clean text from documents
    context_list = [d.page_content.replace("\t", " ") for d in relevant_document_chunks]
    
    #Combine all chunks into a single context string
    context_for_query = "\n\n".join(context_list)
    
    
    #Append conversation history to context
    full_context = context_for_query
    
    #Build prompt using the System message & full context(conversation history + context)
    prompt = f"""[INST]{qna_system_message}
    {{'user': {qna_user_message_template.format(context=full_context, question=user_input)} }}
    [/INST]"""

    OLLAMA_URL = "http://localhost:11434/api/generate"
    
    # Generate a response from the LLaMA model
    response = {

    "prompt" : prompt,         
    "model" : LLM_MODEL,
    "temperature" : 0.06,        #controls randomness of the output  
    "top_p" : 0.90,             #limits token selection to the ones with the mentioned probability
    "max_tokens" : 800,         #max number of tokens the model can generate in the response
    "top_k" : 50,               #limits the number of relevant tokens selection
    "repeat_penalty" : 1.2,     #penalizes repeated phrases in generated text
    "echo" : False,             #prevents the prompt from being repeated in the output
    "stream" : False            #the full response is returned at once instead of one by one

    }

    try:
        #Send a request to the Ollama API with the response in JSON format
         response = requests.post(OLLAMA_URL,json=response)

        #Convert the API response from JSON format into a Python dictionary
         result = response.json()
    #Handle errors
    except Exception as e:
        result = f"Error: {e}"
    
    #return only the generated text
    return result["response"]

In [233]:
questions = [ 
            "What are the effects on the benefits I receive if my probation is extended?",
            "There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?",
            "What should I do if I notice suspected harassment with my female colleague?"
            ]

In [234]:
#looping through each question & corresponding answer
for i,q in enumerate(questions,start=1):
    print(f"\nQuestion {i}: {q}\n")
    response = RAG_with_memory(q)
    print("Answer:")
    print(response)
    print("\n")


Question 1: What are the effects on the benefits I receive if my probation is extended?

Answer:
While on probation, including any extension, you are not eligible for:

* Annual leave encashment
* Internal role transfers
* Performance bonuses

Seniority accrual starts only after successful probation completion. If you have two or more extensions in different roles due to internal transfers, HR will assess contract renewal eligibility based on performance history, and no automatic carry-over is granted.



Question 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

Answer:
To attend the last rites of your demised family member, you can follow these steps:

1. Inform your direct supervisor at least 48 hours before planned leave, except in case of emergencies.
2. Provide live contact details during leave for investigation-related communication.
3. Submit a death certificate, funeral notice,

### **Response Evaluation & Observations**

#### Groundedness - 5/5

- All three responses are strongly grounded in the retrieved context.
- No hallucinated policies, procedures, or external assumptions are introduced.
- Key policy elements such as:
-->Benefits restrictions during probation
-->Leave procedures and eligibility
-->Harassment reporting channels and timelines
are accurately derived from the context.
The model strictly adheres to the context using only derived information, demonstrating high procedural alignment.

The responses are fully grounded with no fabricated information

#### Relevance - 5/5

- All responses directly address the user’s questions without deviation.
- No unnecessary explanations, disclaimers, or unrelated details are included.
- Structured formatting (bullet points / steps) improves clarity and usability.
- Q1 and Q3 are highly precise and complete.
- Q2 includes all major procedural and policy-related aspects relevant to the query.

Highly relevant and focused responses across all queries.

#### Observation

Tuning #8 represents a significant improvement in response quality compared to previous iterations. By removing conversation memory and introducing stricter output constraints, the model demonstrates enhanced groundedness, reduced hallucinations, and better control over response structure.

The outputs are concise, well-structured, and free from prompt leakage issues such as references to input tokens("Context" or "Question") which were observed in earlier tunings. The model effectively prioritizes key policy details and presents them in a clear and actionable format.

A minor omission (document submission timeline) is observed in one of the responses (Q2), this is due to the model prioritizing more important and frequently occurring policies in the retrieved context. We have avoided re-tweaking the prompt in yet another iteration since the other question responses might be affected, leading to regression due to over-tuning.

The overall performance remains strong, with accurate extraction and synthesis of relevant information. The trade-off between completeness and conciseness appears to be well-balanced.

# **CRITERIA 6: Comparitive,Actionable & Business Insights**

## **Pre-Tuning Performances**

### **Using LLM Only**

- As the model produced hallucinations such as interpreting the scenario as **criminal probation**, there was **no grounding** in the response.
- Since the model did not have access to any company-specific information, the response showed **no policy alignment** with the employee handbook.
- The model was able to provide **some relevance** in simple cases, as it could still understand general intent from the question despite lacking context.
- **Actionable Insight:** The absence of context grounding highlights the **need to integrate a retrieval mechanism** for policy accuracy.

### **Using LLM + Prompt Engineering**

- With the introduction of prompts, the model demonstrated **improved relevance** by better understanding HR-related scenarios.
- However, due to the absence of retrieved documents, the **responses remained generic** and lacked policy-specific details.
- **Actionable Insight:** While prompt engineering improves relevance, it **must be combined with external knowledge sources to avoid generic responses**.

### **RAG Setup**

- The implementation of RAG enabled **proper grounding**, as the model now had access to relevant document chunks.
- The retriever successfully **fetched relevant context**, ensuring that the system had the necessary information to answer queries.
- However, the quality of the final response still depended on how effectively the model utilized the retrieved information.
- **Actionable Insight:** Effective retrieval alone is insufficient, and so we **need to optimize the generation step to utilize retrieved context** for performance.

### **RAG + Memory + Initial Prompt**

- The use of RAG significantly **improved groundedness**, as responses were now based on retrieved policy documents.
- The model achieved **high relevance**, as it was able to correctly answer user queries using contextual information.
- However, the model exhibited **partial extraction of details**, as some important policy points were still missed.
- Additionally, the responses **included generic disclaimers**, indicating that the **model was still relying on pre-trained patterns** instead of strictly following the context.
- Introducing **conversational memory** showed that the model is capable of **retaining information from previous interactions**, enabling continuity across related queries.
- Follow-up questions confirmed that the model could recall and reuse prior responses, indicating effective conversational memory capability.
- **Actionable Insight:** Prompt refinement **needs to improve complete information extraction** and **reduce generic LLM behaviors** such as disclaimers.

## **Tuning Performances**

### **Tuning#1**

- By increasing the **temperature from 0.01 to 0.05**, the model **reduced response rigidity**, allowing it to generate more natural and flexible answers.
- This change improved the **inclusion of missing policy details**, as the model was able to explore slightly broader variations in the retrieved context.
- Despite this adjustment, the model maintained **strong groundedness and relevance**, as it continued to rely on retrieved information with **k = 6** unchanged.
- **Actionable Insight:** Slightly **increasing temperature can reduce rigidity** and improve detail coverage without compromising grounding.

### **Tuning#2**

- By further increasing the temperature from **0.05 to 0.1** and introducing **top_p = 0.90**, the model achieved a **better balance between completeness and precision**.
- The model was able to include more policy-specific details, **improving overall answer quality**.
- However, this also led to the **inclusion of minor irrelevant elements**, indicating a trade-off introduced by increased randomness.
- **Actionable Insight:** **Controlled randomness** using temperature and top_p helps achieve an optimal balance between completeness and precision.

### **Tuning#3**

- By reducing the temperature from **0.1 to 0.08** and introducing stricter prompt constraints, the model successfully **reduced hallucinations**.
- However, the model became over-restricted, leading to the **omission of key details** despite having k = 6 and top_p = 0.90 unchanged.
- In some cases, the model generated unsupported or **incorrect suggestions**, showing the negative impact of excessive constraint.
- **Actionable Insight:** Over **strict prompts should be avoided** as they can reduce completeness and lead to loss of important information.

### **Tuning#4**

- With the temperature = 0.08 and top_p = 0.90, we **refined the prompt** to balance control and flexibility, improving response behavior.
- This resulted in completeness and structure, as the model was better able to **extract relevant details** from retrieved chunks.
- However, issues such as **email-style formatting** and **repetitive content** persisted, indicating that prompt refinement alone was not sufficient.
- **Actionable Insight:** Prompt design **should aim to balance structure and flexibility** to improve both readability and completeness.

### **Tuning#5**

- With temperature = 0.08, top_p = 0.90, and k = 6 unchanged, we introduced further **prompt simplification** and **control instructions**.
- This improved response **clarity and reduced redundancy**, leading to more **structured outputs**.
- However, retrieval inconsistencies still caused **context mixing** and occasional **irrelevant responses**, highlighting limitations beyond prompt tuning.
- **Actionable Insight:** Prompt improvements alone cannot resolve issues caused by retrieval inconsistencies, highlighting the **need for in depth tuning**

### **Tuning#6**

- By increasing the retrieval parameter from **k = 6 to k = 8**, while keeping temperature = 0.08 and top_p = 0.90 constant, the model **accessed more context**.
- This **improved context coverage**, but also **introduced noise and irrelevant details** from additional chunks.
- As a result, the model struggled with prioritization, leading to a **decrease in overall answer quality**.
- We have figured out the **root cause** for the hallucinating responses and inaccuracies. While memory retention improved the recall and reuse of previous responses, it also led to context mixing and unintended carryover, where information from previous queries influenced unrelated responses.
- **Actionable Insight:** We need to **remove conversational memory** and be **cautious while increasing retrieval size**, as excessive context can introduce noise and reduce answer quality.

### **Tuning#7**

- By reducing the temperature from **0.08 to 0.05** and **removing conversational memory**, the model **improved response clarity** and **reduced contextual noise**.
- After resetting **k = 6**, the model became more focused on the current query, **improving relevance** in most cases.
- However, the model occasionally failed to utilize retrieved context effectively, resulting in **generic or fallback responses**.
- Removing memory **improved response independence**, ensuring that each query was answered based on the current question alone.
- **Actionable Insight:** **Removing conversational memory** can improve response focus, but a **stronger prompt** is required to ensure proper context usage.

### **Tuning#8**

- By slightly increasing the **temperature from 0.05 to 0.06** and introducing **stricter output constraints** in the prompt, the model achieved **optimal performance**.
- The model ensured **complete context grounding** and included all relevant policy details, **improving overall completeness**.
- Issues such as **hallucinations, disclaimers, and formatting inconsistencies were eliminated** while keeping k = 6 and top_p = 0.90 constant.
- This resulted in **consistently accurate, relevant, and well-structured responses**, making this the **best-performing tuning iteration**.
- **Actionable Insight:** Enforcing strict context usage and output constraints is key to achieving consistent, high-quality responses in RAG systems.

## **Business Insights**

- By implementing RAG into the system, we significantly **improved policy adherence** and **reduced the risk of incorrect responses** in enterprise applications
- Fine-tuning model parameters such as **temperature**, **top_p** & **retrieval size** directly **impacts the accuracy and reliability** of responses, making them critical for production systems.
- **Excessively strict or poorly designed prompts** can result in **confusion and noise**, which negatively affects the users' trust in AI-driven systems.
- Removing **unnecessary memory conversation**  & disclaimers **enhances clarity and ensures** more focused and actionable responses.
- An optimised RAG system can be a **reliable decision supporting tool** for employees by providing **accurate & context-aware policy guidance**.
- The study demonstrates that achieving high performance requires a **balance between retrieval quality, prompt design & parameter tuning**, rather than optimizing a single component.
- Also, conversational memory can enhance user experience in multi-turn interactions by **enabling continuity and contextual awareness** across queries. However, while memory is beneficial, it may not be suitable for policy based Q&A systems. For policy driven systems, a **stateless design** ensures **more consistent, reliable, and contextually accurate responses**.

# **CONCLUSION**

The development of the RAG based question & answering system demonstrates a clear progression from ungrounded and generic responses to highly accuratE & relevant outputs. Our initial approach without RAG resulted in hallucinated & irrelevant answers, while the introduction of **RAG significantly improved both groundedness & relevance** by enabling access to the policy specific information.

Further, tuning iterations highlighted the importance of balancing multiple factors like prompt design, parameter tuning, & retrieval configuration. While **increasing temperature improved completeness**, **excessive constraints in prompt design led to loss of critical details**, & **higher retrieval values introduced noise**. Removing conversational memory further **improved response clarity** by eliminating unnecessary context interference.

Also, experiments with **conversational memory** demonstrated that the model is capable of retaining and utilizing information from previous interactions which **enabled continuity across queries**. However, this also introduced **context mixing and unintended carryover**, which negatively impacted accuracy in unrelated questions. So, **removing memory improved response relevance** by ensuring that each query was processed independently. From this we can conclude that while conversational memory is beneficial for multi-turn applications, a **stateless approach is more suitable for structured & policy driven systems**.

The final tuning achieved optimal performance by **enforcing strict context usage**, **ensuring complete extraction of relevant details**, and **eliminating hallucinations** and **formatting inconsistencies**. This resulted in consistently accurate, relevant, and well-structured responses across all queries.

To conclude, this project shows that an effective RAG system requires a careful **balance between retrieval quality, prompt engineering, and parameter tuning**, and that **refinement in iterations** is necessary to achieve **reliable and production-ready performance**.

# **ADDITIONAL: FRONT END/API INTEGRATION**

During the development and tuning phase, we used the locally hosted model **LLaMA 2 13B** via **Ollama**  to enable efficient experimentation, full control over parameters, and faster iteration without dependency on external APIs. This allowed multiple tuning iterations to be performed reliably while maintaining consistency in evaluation.

For frontend deployment, we choose to use the hosted model **Mistral-7B-Instruct via Hugging Face** due to its accessibility and compatibility with cloud based interfaces. Since it isn't possible to directly access locally hosted models in deployed environments, this model change will ensure seamless integration without altering the core RAG pipeline.

Most importantly, **the retrieval mechanism, the prompt design, and the evaluation framework will remain unchanged** across both setups, ensuring that the observed performance improvements are due to the system design rather than the specific model used.

In RAG systems, **prompt design and retrieval quality have a greater impact on performance than the choice of base model**.

## **Setup Hugging Face Client & Model Initialization**

In [50]:
!pip install huggingface_hub


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
#Import InferenceClient to interact with Hugging Face hosted models
from huggingface_hub import InferenceClient
import os

import json
import tiktoken 

import pandas as pd

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings
)
from langchain_community.vectorstores import Chroma

C:\Users\tvana\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [575]:
!pip install flask==2.2.2 gunicorn==20.1.0 langchain chromadb openai tiktoken

     ---------------------------------------- 0.0/101.5 kB ? eta -:--:--
     ----------------------------------- --- 92.2/101.5 kB 2.6 MB/s eta 0:00:01
     -------------------------------------- 101.5/101.5 kB 1.9 MB/s eta 0:00:00
     ---------------------------------------- 0.0/79.5 kB ? eta -:--:--
     ---------------------------------------- 79.5/79.5 kB ? eta 0:00:00
     ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
     -------------------- ------------------- 0.6/1.1 MB 12.2 MB/s eta 0:00:01
     ---------------------------------------- 1.1/1.1 MB 18.3 MB/s eta 0:00:00
     ---------------------------------------- 0.0/226.3 kB ? eta -:--:--
     ------------------------------------- 226.3/226.3 kB 13.5 MB/s eta 0:00:00
     ---------------------------------------- 0.0/206.0 kB ? eta -:--:--
     ---------------------------------------- 206.0/206.0 kB ? eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [58]:
#Store Hugging Face token
os.environ["HF_TOKEN"] = "actual_token"

In [85]:
#Initialize the client with Mistral model

client = InferenceClient(
    model="mistralai/Mistral-7B-Instruct-v0.3",
    token=os.getenv("HF_TOKEN")                    #Authentication token for API access
)

## **Re-initializing Variables For ease of use**

In [22]:
pdf_file = "Dataset - Flykite Airlines_HRP.pdf"

In [23]:
#loading the file using PyPDFLoader (since there's just one file)
pdf_loader = PyPDFLoader(pdf_file)

In [24]:
#Load the contents of the PDF into a list of documents
documents = pdf_loader.load()

In [25]:
#Print the no of pages loaded
print("Number of pages:",len(documents))

Number of pages: 14


In [26]:
#Initializing the text splitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       #chunk size is a bit high since smaller chunks may split important policy explanations to multiple chunks
    chunk_overlap=100     #overlap is also high so that the context is retained between chunks containing important policies
)

In [27]:
#Load the PDF and split it into chunks simultaneously
document_chunks = pdf_loader.load_and_split(text_splitter)

In [28]:
#Printing the no of chunks created.
print("Number of chunks:",len(document_chunks))

Number of chunks: 34


In [29]:
#Looking at the first chunk

print(document_chunks[0])

page_content='Flykite\n \nAirlines:\n \nHuman\n \nResources\n \nPolicy\n \nHandbook\n \nIntroduction\n \nFlykite\n \nAirlines\n \nis\n \ndedicated\n \nto\n \ncultiv ating\n \nan\n \norganizational\n \ncultur e\n \nthat\n \nsyner gizes\n \noper ational\n \nexcellence\n \nwith\n \na\n \nsuppor tive,\n \nequitable,\n \nand\n \nlegally\n \ncompliant\n \nworkplace\n \nenvir onment\n \nacross\n \nall\n \ndepar tments\n \nand\n \nemplo yee\n \nlevels.\n \nThis\n \ndocument\n \nestablishes\n \nan\n \nexhaustiv e\n \nframework\n \ncomprising\n \nall\n \nhuman\n \nresour ce\n \npolicies\n \ncurrently\n \nin\n \neffect.\n \nAll\n \nprovisions\n \nare\n \nsubject\n \nto\n \namendment,\n \ninterpr etation,\n \nor\n \nrescindment\n \nat\n \nthe\n \nsole\n \ndiscr etion\n \nof\n \nthe\n \nHuman\n \nResour ces\n \nand\n \nLegal\n \ndepar tments.\n \nIn\n \nthe\n \nevent\n \nof\n \nambiguities\n \nor\n \nconﬂicting\n \ninterpr etations,\n \nthe\n \noﬃcial' metadata={'source': 'Dataset - Flykite Airline

In [30]:
#Clean newline characters in each chunk
for doc in document_chunks:
    doc.page_content = doc.page_content.replace("\n", " ")

In [31]:
#Re-checking the first chunk

print(document_chunks[0])

page_content='Flykite   Airlines:   Human   Resources   Policy   Handbook   Introduction   Flykite   Airlines   is   dedicated   to   cultiv ating   an   organizational   cultur e   that   syner gizes   oper ational   excellence   with   a   suppor tive,   equitable,   and   legally   compliant   workplace   envir onment   across   all   depar tments   and   emplo yee   levels.   This   document   establishes   an   exhaustiv e   framework   comprising   all   human   resour ce   policies   currently   in   effect.   All   provisions   are   subject   to   amendment,   interpr etation,   or   rescindment   at   the   sole   discr etion   of   the   Human   Resour ces   and   Legal   depar tments.   In   the   event   of   ambiguities   or   conﬂicting   interpr etations,   the   oﬃcial' metadata={'source': 'Dataset - Flykite Airlines_HRP.pdf', 'page': 0}


In [32]:
import re

for doc in document_chunks:
    text = doc.page_content

    # Remove single-character newlines (main issue)
    text = re.sub(r'(?<=\w)\n(?=\w)', '', text)

    # Replace remaining newlines with space
    text = text.replace("\n", " ")

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # Fix spacing before punctuation
    text = re.sub(r'\s+([.,:;])', r'\1', text)

    doc.page_content = text.strip()

In [33]:
#also checking consecutive chunk
print(document_chunks[1])

page_content='In the event of ambiguities or conﬂicting interpr etations, the oﬃcial determinations by these depar tments shall prevail and govern subsequent actions. 1. Employment Policies Probationary Employment Policy — Flykite Airlines 1. Duration of Initial Probation ● All new emplo yees are placed on a probationar y period of 90 calendar days from their oﬃcial start date. ● For technical, safety-critical, or senior management roles, probation is 120 calendar days. ● Any probation may be extended only once, for a maximum of 90 additional days, provided that a Performance Impr ovement Plan (PIP) is appr oved by' metadata={'source': 'Dataset - Flykite Airlines_HRP.pdf', 'page': 0}


In [34]:
#choosing the GTE embedding model since it's designed for semantic retrieval task
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

print("The embedding model has loaded successfully")

The embedding model has loaded successfully


In [75]:
#defining name for the Chroma collection
report = 'Flykite_Airlines_HRP'

#Each doc chunk is converted into an embedding vector which is then stored in Chroma database
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    collection_name=report
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [76]:
#testing if the4 vector database got created
print(vectorstore)

In [77]:
#Converts the Chroma vector store into a retriever for querying.
#Define the retriever with an appropriate search method and k value 
retriever = vectorstore.as_retriever(
    search_type='similarity',     # Specifies that retrieval is based on cosine similarity 
    search_kwargs={'k': 10}        # Retrieves the top 3 most similar documents for a given query.
)

In [47]:
qna_system_message = """
You are a strict extraction assistant.

You MUST ONLY use the provided context.

RULES:
- Extract ALL information relevant to the question.
- Do NOT skip any relevant detail, even if it appears minor or repetitive.
- Ensure the answer fully covers all parts of the question.
- Do NOT add any information not present in the context.
- Do NOT assume or infer anything.
- If the information is not explicitly present in the context, do NOT include it.
- Do NOT provide your personal opinion or any generic statement.

OUTPUT RULES:
- Return ONLY bullet points.
- No introductions.
- No conclusions.
- No explanations.
- No source references.
- Each bullet must be directly supported by the context.
"""

In [48]:
qna_user_message_template = """
Context:
{context}

Question:
{question}

Task:
Extract ALL relevant information from the context that directly answers the question.
Ensure no important detail is missed.

Return only bullet points.
"""

## Response Generation Function with Mistral

In [74]:
def RAG_without_memory(user_input):
    """
    Args:
        user_input: User query
        llm: Language model
    
    Returns:
        Generated response using RAG
    """

    
#Removes irrelevant and repetitive legal or disclaimer text from retrieved chunks
    def clean_context(chunks):
        cleaned = []
        for c in chunks:
            text = c.page_content
        
        # Remove only noisy lines, not full chunk
            for phrase in [
                "liable for legal action",
                "this file is meant for personal use",
                "do not share",
                "confidential"
            ]:
                text = text.replace(phrase, "")
                
            c.page_content = text
            cleaned.append(c)
    
        return cleaned

    
    # Retrieve documents
    docs = vectorstore.similarity_search(user_input, k=6)

    # Take top 3 most relevant chunks
    relevant_document_chunks = docs[:6]

    #Clean the doc chunks
    relevant_document_chunks = clean_context(relevant_document_chunks)
    
    #Extract and clean text from documents
    context_list = [d.page_content.replace("\t", " ") for d in relevant_document_chunks]
    
    #Combine all chunks into a single context string
    context_for_query = ""
    
    for i, chunk in enumerate(context_list):
        context_for_query += f"""
    [Source {i+1}]
    {chunk}
"""
    
    
    #Append conversation history to context
    full_context = ""

    for i, chunk in enumerate(context_list):
        full_context += f"""
    [Source {i+1}]
    {chunk}
    
"""
    
    # Generate a response from the LLaMA model
    response = client.chat_completion(
        messages= [{"role" : "system","content" : qna_system_message},
                   {"role" : "user","content" : qna_user_message_template.format(
                    context = full_context,
                    question=user_input)}
                    ],
        temperature = 0.05,                               #controls randomness of the output  
        max_tokens=400,                                   #max number of tokens the model can generate in the response
        top_p = 0.9                                       #limits token selection to the ones with the mentioned probability
    )

    result = response.choices[0].message["content"]

    #return the result
    return result

In [50]:
questions = [ 
            "What are the effects on the benefits I receive if my probation is extended?",
            "There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?",
            "What should I do if I notice suspected harassment with my female colleague?"
            ]

In [35]:
#looping through each question & corresponding answer
for i,q in enumerate(questions,start=1):
    print(f"\nQuestion {i}: {q}\n")
    response = RAG_without_memory(q)
    print("Answer:")
    print(response)
    print("\n")


Question 1: What are the effects on the benefits I receive if my probation is extended?



Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Answer:
● During probation extension, emplo yees are not eligible for: ○ Annual leave encashment ○ Internal role transfers ○ Performance bonuses
● Seniority accrual starts only after successful probation completion.
● If an emplo yee has two or more extensions in different roles, HR will assess contr act renewal eligibility based on performance history. No automatic carry-over is granted.



Question 2: There has been a demise in my family last night, and I need to attend the last rites.How should I inform the office, and will I be granted leave?

Answer:
● Contact your direct supervisor as soon as possible, preferably at least 1 day before your absence.
● Provide details about the family demise and the need to attend the last rites.
● Inform your supervisor about the expected duration of your absence.
● Request for leave approval and provide any necessary documentation, such as a death certificate or funeral notice.
● Follow up with your supervisor to confirm leave approval and any ad

### Observation

The transition from LLaMA to Mistral was not seamless because different models have different response behaviors.

Although both are instruction tuned models, Mistral tends to:

* Be more verbose and helpful
* Add general knowledge even when not present in the context

As a result, the prompt and parameters that worked for LLaMA did not work much with Mistral. The model began generating more generic and hallucinated responses.

To address this, we had to:

* Refine the prompt to enforce stricter rules
* Adjust parameters to reduce creativity and improve determinism

This retuning ensured that Mistral aligned with the goal of the system & generated context-based answers.


## **App Backend**

In [29]:
#importing necessary libraries
!pip install werkzeug==2.2.3

     ---------------------------------------- 0.0/233.6 kB ? eta -:--:--
     --------------------------------- ---- 204.8/233.6 kB 6.1 MB/s eta 0:00:01
     -------------------------------------- 233.6/233.6 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: werkzeug
    Found existing installation: Werkzeug 3.1.7
    Uninstalling Werkzeug-3.1.7:
      Successfully uninstalled Werkzeug-3.1.7



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Create the main RAG pipeline file

In [51]:
# Create a folder for storing the files needed for backend server deployment
import os
os.makedirs("backend_files", exist_ok=True)

In [2]:
#Store Hugging Face token
os.environ["HF_TOKEN"] = "actual_token"

NameError: name 'os' is not defined

In [90]:
%%writefile backend_files/rag_pipeline.py
import requests
import json
import tiktoken 
import re
import pandas as pd
import time

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings
)
from langchain_community.vectorstores import Chroma

pdf_file = "Dataset - Flykite Airlines_HRP.pdf"

#loading the file using PyPDFLoader (since there's just one file)
pdf_loader = PyPDFLoader(pdf_file)

embedding_model = SentenceTransformerEmbeddings(
    model_name='thenlper/gte-large'
)

#Initializing the text splitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       #chunk size is a bit high since smaller chunks may split important policy explanations to multiple chunks
    chunk_overlap=100     #overlap is also high so that the context is retained between chunks containing important policies
)

document_chunks = pdf_loader.load_and_split(text_splitter)

#defining name for the Chroma collection
report = 'Flykite_Airlines_HRP'

vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    collection_name=report
)

def retrieve_docs(query):
    docs = vectorstore.similarity_search(query, k=3)
    return "\n\n".join([doc.page_content for doc in docs])

#HF ENDPOINT
API_URL = "https://w679jec3hq4g863h.us-east-1.aws.endpoints.huggingface.cloud/invocations"
HEADERS = {
    "Authorization": "Bearer actual_token"
}

#Your system prompt (keep your tuned one)
qna_system_message = """You are a strict extraction assistant.

You MUST ONLY use the provided context.

RULES:
- Extract ALL information relevant to the question.
- Do NOT skip any relevant detail, even if it appears minor or repetitive.
- Ensure the answer fully covers all parts of the question.
- Do NOT add any information not present in the context.
- Do NOT assume or infer anything.
- If the information is not explicitly present in the context, do NOT include it.
- Do NOT provide your personal opinion or any generic statement.

OUTPUT RULES:
- Return ONLY bullet points.
- No introductions.
- No conclusions.
- No explanations.
- No source references.
- Each bullet must be directly supported by the context.
"""






def query(payload):
    for _ in range(3):  # retry 3 times
        response = requests.post(API_URL, headers=HEADERS, json=payload)
        result = response.json()

        if "error" not in result:
            return result

        print("Retrying due to:", result)
        time.sleep(3)  # wait before retry

    return result  # return last attempt even if failed


def RAG_without_memory(user_input):
    print("RAG FUNCTION CALLED")

    #retrieval added
    full_context = retrieve_docs(user_input)

    prompt = f"""[INST]
{qna_system_message}

Context:
{full_context}

Question:
{user_input}

Answer:
[/INST]"""

    print("PROMPT:\n", prompt)

    response = query({
    "model": "mistralai/Mistral-7B-Instruct-v0.3",
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ],
    "max_tokens": 400,
    "temperature": 0.06
})

    print("RAW RESPONSE:", response)

    return response["choices"][0]["message"]["content"]

Overwriting backend_files/rag_pipeline.py


### Flask API Setup

In [91]:
%%writefile backend_files/app.py

#Import required libraries
from flask import Flask, request, jsonify
from rag_pipeline import RAG_without_memory 

#Initialize Flask app
app = Flask("RAG_QnA_API")


#Home route 
@app.get('/')
def home():
    return "Welcome to the RAG Q&A API!"


#API endpoint for question answering
print("=== REQUEST RECEIVED ===")

@app.route('/ask', methods=['POST'])
def ask_question():
    print(" HIT /ask endpoint")

    

    try:
        print("Headers:", request.headers)
        print("Raw data:", request.data)
        # SAFEST way to get JSON
        data = request.get_json(silent=True)
        print("DEBUG data:", data)

        if not data or "question" not in data:
            return jsonify({"error": "Invalid input"}), 400

        user_question = data["question"]
        print("DEBUG question:", user_question)

        answer = RAG_without_memory(user_question)
        print("DEBUG answer:", answer)

        return jsonify({
            "question": user_question,
            "answer": answer
        })

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500


#Run the app
if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8000)

Overwriting backend_files/app.py


### Dependencies File

In [102]:
%%writefile backend_files/requirements.txt
flask==2.2.2
gunicorn==20.1.0


langchain
chromadb
openai
tiktoken

Overwriting backend_files/requirements.txt


### Dockerfile

In [103]:
%%writefile backend_files/Dockerfile
FROM python:3.9-slim

# Set the working directory inside the container
WORKDIR /app

# Copy all files from the current directory to the container's working directory
COPY . .

# Install dependencies from the requirements file without using cache to reduce image size
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Run Flask app using gunicorn
CMD ["gunicorn", "-w", "2", "-b", "0.0.0.0:8000", "app:app"]

Overwriting backend_files/Dockerfile


### Install Dependencies

In [44]:
# Install all required dependencies from the generated requirements file
!pip install -r backend_files/requirements.txt


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## **App Frontend**

### Streamlit for Interactive UI

In [76]:
# Create a folder for storing the files needed for frontend UI deployment
os.makedirs("frontend_files", exist_ok=True)

In [77]:
%%writefile frontend_files/app.py

import requests
import streamlit as st

st.title("RAG Q&A System")

# Input box
user_question = st.text_input("Enter your question:")

# Button
if st.button("Get Answer"):
    if user_question:
        try:
            response = requests.post(
                "http://127.0.0.1:8000/ask",
                json={"question": user_question}
            )
            
            if response.status_code == 200:
                st.write("Answer:")
                st.write(response.json()["answer"])
            else:
                st.write("Error from backend")
        
        except:
            st.write("Backend not running. Please start Flask API.")

Overwriting frontend_files/app.py


# **Deployed Application**

The RAG Q&A system has been deployed locally using Flask + Streamlit.

**Access URL:**
http://localhost:8501

**Instructions to run:**

1. Start backend(in terminal):
   python app.py
   
2. Start frontend:
   streamlit run app.py
   
3. Open browser at the above URL.


In [1]:
%%writefile audit.md

# Project Audit – Vanathi Airlines RAG
_Last updated: April 2026_

This document evaluates the current state of the Airlines RAG project from a system design and production readiness perspective, 
identifying gaps and outlining improvements to reach production-grade GenAI system standards.

## 1. Current Functionality
- Airline Q&A chatbot using Retrieval-Augmented Generation (RAG)
- Handles queries on flights, bookings, policies
- Pipeline: Embedding → Vector Search → LLM → Response
- Built using LangChain + OpenAI + ChromaDB

---

## 2. Tech Stack
- Python 3.x
- LangChain (pre-LCEL usage)
- OpenAI API (GPT models)
- ChromaDB (vector store)
- Jupyter Notebook (current)
- Optional Streamlit/Gradio UI

---

## 3. System Design Gaps (Senior-Level View)

### Scalability
- No batching or async handling
- No caching (LLM responses / embeddings)
- No horizontal scaling strategy

### Reliability
- No retry mechanisms (LLM/API failures)
- No fallback models
- No circuit breaker or timeout handling

### Observability
- No tracing (LangSmith)
- No logs for:
  - latency
  - token usage
  - failures
- No alerting

### Cost Awareness
- No tracking of token usage
- No cost optimization (caching, model selection)

---

## 4. Retrieval & RAG Gaps

### Retrieval Quality
- Only semantic similarity (no hybrid BM25)
- No re-ranking layer
- No metadata filtering

### Context Quality
- Fixed chunking (no semantic chunking)
- No query rewriting
- No multi-hop retrieval

---

## 5. LLM / Generation Gaps

- No structured prompting (few-shot / templates)
- No hallucination control (grounded generation missing)
- No response validation layer
- No streaming responses

---

## 6. Architecture Gaps

- Notebook-based (non-modular)
- No API layer (FastAPI missing)
- No separation of:
  - retrieval
  - generation
  - evaluation
- Hardcoded configs (no env/config management)

---

## 7. Evaluation Gaps (CRITICAL)

- No automated evaluation
- No defined metrics:
  - Faithfulness
  - Answer relevance
  - Context recall
- No dataset for testing queries

---

## 8. Failure Modes (VERY IMPORTANT)

System may fail in cases like:
- Irrelevant retrieval → hallucinated answer
- Long/complex queries → incomplete context
- Ambiguous queries → wrong interpretation
- API failure → no fallback
- High latency → poor UX

---

## 9. Success Metrics (Add This for Interviews)

Target metrics after improvements:

- Faithfulness score > 0.85 (RAGAS)
- Answer relevance > 0.85
- Latency < 2.5 seconds
- Retrieval accuracy (top-k hit rate) > 80%
- Cost per query minimized via caching

---

## 10. Improvement Roadmap

### Phase 1 (Week 1–2): Foundation & Retrieval
- Migrate to LCEL + modular architecture
- Implement hybrid search (BM25 + embeddings)
- Add re-ranking (Cohere / cross-encoder)
- Introduce RAGAS evaluation

### Phase 2 (Week 2–3): Production Layer
- Build FastAPI backend
- Add streaming responses
- Add logging (latency, tokens, errors)
- Integrate LangSmith tracing
- Add retry + fallback mechanisms

### Phase 3 (Week 6–7): Deployment
- Dockerize application
- Deploy (Render / AWS / Azure)
- Add monitoring dashboard
- Write production-grade README + architecture diagram

---

## Final Goal

Transform from:
 "Basic RAG notebook project"

To:
"Production-ready, evaluated, observable RAG system with scalable architecture"

This demonstrates:
- System design thinking
- LLM application engineering
- Production readiness

Writing audit.md
